In [ ]:
import os
import matplotlib.pyplot as plt
import cv2
import image_process_tool_box as iat
import numpy as np
import random as rand
from speckle_pattern import generate_and_save
from scipy import integrate
import math
import torch
import matplotlib.pyplot as plt




# 1. Miscellenuous 




#### FE-deform


In [ ]:
import numpy as np
import file_paths as path
import matplotlib.pyplot as plt

op2_path = path.hole_5_op
bdf_path = path.hole_5_bdf

def generate_and_deform_analytical(
        BDF_path,
        OP2_path,
        image_height, image_width,
        perlin_dictionary,
        showmesh=False,
        texture=None, 
        difference_image = False):
    
    # Load required libraries
    import image_process_tool_box as ipt
    import matplotlib.pyplot as plt
    from pyNastran.bdf.bdf import BDF
    from pyNastran.op2.op2 import OP2
    from noise import pnoise2
    import numpy as np
    from perlin_noise import PerlinNoise



    # Get FE data and scale to image
    nodes_2d, deformed_nodes_2d = ipt.load_fe_nodes(BDF_path,OP2_path,showmesh)
    max_x_FE = (np.max(nodes_2d[:,0]))
    x_scale = image_width / max_x_FE  
    original_points = nodes_2d * x_scale
    new_points = deformed_nodes_2d * x_scale

    # Generate interpolators
    _, _, FEx_rbf_interp, FEy_rbf_interp = ipt.smooth_field(
            np.zeros((image_height, image_width)),
            original_points,
            new_points,
            3
        )
    
    # Generate and deform perlin noise images
    y, x = np.meshgrid(np.arange(image_height), np.arange(image_width), indexing='ij')
    coords = np.stack([x.flatten(), y.flatten()], axis=-1)

    
    # Get displacement fields. Evaluated at coordinate array locations.
    dx_vals = FEx_rbf_interp(coords).reshape(image_height, image_width)
    dy_vals = FEy_rbf_interp(coords).reshape(image_height, image_width)

    # Create the inverse displacement function d⁻¹(x,y)
    # For each point (x,y) in the deformed image, d⁻¹(x,y) gives the position in the original space
    x_inv = x - dx_vals
    y_inv = y - dy_vals
    
    for sample_idx in range(len(perlin_dictionary[:,0])):

        # Generate template image using Perlin noise
        template = np.zeros((image_height, image_width), dtype=np.float32)
        for row in range(image_height):
            for col in range(image_width):      
                # Sample noise at each integer pixel location
                template[row, col] = pnoise2(col * perlin_dictionary[sample_idx,0], row * perlin_dictionary[sample_idx,0],
                                        octaves=int(round(perlin_dictionary[sample_idx,1])),
                                        persistence=perlin_dictionary[sample_idx,2],
                                        lacunarity=perlin_dictionary[sample_idx,3],
                                        repeatx=image_width, repeaty=image_height,
                                        base=0)
        # time.sleep(0.75)

        # Generate deformed image using the inverse mapping: 
        # new_pixel_x, new_pixel_y = noise(d⁻¹(old_pixel_x, old_pixel_y))
        deformed = np.zeros((image_height, image_width), dtype=np.float32)
        for row in range(image_height):
            for col in range(image_width):
                # Get the inverse mapped point by applying inverse-deformed grid coordinates
                # when sampling the Perlin noise.
                # Because of inverse deformation, the pixel locations may not be integers anymore
                # but thats fine as the noise can be sampled from anywhere in 2D space
                src_x = x_inv[row, col] * perlin_dictionary[sample_idx,0]
                src_y = y_inv[row, col] * perlin_dictionary[sample_idx,0]
                
                # Evaluate the noise function at the inverse mapped point
                deformed[row, col] = pnoise2(src_x, src_y,
                                      octaves=int(round(perlin_dictionary[sample_idx,1])),
                                        persistence=perlin_dictionary[sample_idx,2],
                                        lacunarity=perlin_dictionary[sample_idx,3],
                                        repeatx=image_width, repeaty=image_height,
                                        base=0)
                

        
        # Normalise images to [0, 255]
        # As unsigned integer 8-bit
        template = ((template - template.min()) / (template.max() - template.min()) * 255).astype(np.uint8)
        deformed = ((deformed - deformed.min()) / (deformed.max() - deformed.min()) * 255).astype(np.uint8)

        # Apply texture functions
        textured_template = ipt.apply_texture(template, texture_type=texture, scale=texture, seed=None)
        textured_deformed = ipt.apply_texture(deformed, texture_type=texture, scale=texture, seed=None)

        # Difference image
        original_float = textured_template.astype(np.float32)
        remapped_float = textured_deformed.astype(np.float32)
        diffim = np.abs(original_float - remapped_float)

        return textured_template,textured_deformed, diffim

perlin_dict = np.array([[
    0.05,    # scale
    3,       # octaves
    1.0,     # persistence
    2        # lacunarity
]])

ref,deform,differ = generate_and_deform_analytical(
    bdf_path,
    op2_path,
    500,2000,
    perlin_dict,showmesh=True
)

plt.imshow(ref)
plt.show()
plt.imshow(deform)
plt.show()
plt.imshow(differ,cmap='jet')
plt.show()

collapsed_difference = np.mean(differ, axis=0)
x_line = np.arange(differ.shape[1])

plt.plot(x_line,collapsed_difference)
plt.show()



In [ ]:
import cv2
import numpy as np
import file_paths as path
import matplotlib.pyplot as plt

p1 = r"data\speckle_pattern_img\reference_im\2_Generated_spec_image.tif"
p2 = r"data\speckle_pattern_img\deformed_im\3_Generated_spec_image.tif"
ref = cv2.imread(p1,cv2.IMREAD_GRAYSCALE)
deform  =  cv2.imread(p2,cv2.IMREAD_GRAYSCALE)

plt.imshow(ref)
plt.show()
plt.imshow(deform)
plt.show()

original_float = ref.astype(np.float32)
remapped_float = deform.astype(np.float32)
diffim = np.abs(original_float - remapped_float)

collapsed_difference = np.mean(diffim, axis=0)
x_line = np.arange(diffim.shape[1])

plt.plot(x_line,collapsed_difference)
plt.show()

In [ ]:
import cv2
import numpy as np
import file_paths as path
import matplotlib.pyplot as plt

plt.rcParams.update({
    'font.family': 'Calibri',
    'font.size': 26,
    'axes.labelsize': 26,
    'xtick.labelsize': 18,
    'ytick.labelsize': 18,
    'legend.fontsize': 14,
    'lines.linewidth': 1.5,
    'lines.markersize': 3,

    # Border styling
    'axes.linewidth': 2,     
    'xtick.major.width': 2,    
    'ytick.major.width': 2,    
    'xtick.minor.width': 1.1,    
    'ytick.minor.width': 1.1,

    'xtick.major.size': 10,      
    'ytick.major.size': 10,
    'xtick.minor.size': 6,      
    'ytick.minor.size': 6,

    'xtick.minor.visible': True,
    'ytick.minor.visible': True,
    'xtick.top': True,          
    'xtick.bottom': True,    
    'ytick.left': True,        
    'ytick.right': True,       
    'xtick.direction': 'in', 
    'ytick.direction': 'in',

})
indddd = [0,2,10]

for inw in indddd:
    p1 = rf"data\speckle_pattern_img\reference_im\{inw}_Generated_spec_image.tif"
    p2 = rf"data\speckle_pattern_img\deformed_im\{inw+1}_Generated_spec_image.tif"
    ref = cv2.imread(p1,cv2.IMREAD_GRAYSCALE)
    deform  =  cv2.imread(p2,cv2.IMREAD_GRAYSCALE)

    # plt.imshow(ref)
    # plt.show()
    # plt.imshow(deform)
    # plt.show()
    original_float = ref.astype(np.float32)
    remapped_float = deform.astype(np.float32)
    diffim = np.abs(original_float - remapped_float)
    plt.figure(figsize=(12,8))
    plt.imshow(diffim, cmap='jet', vmin=0, vmax=255)
    cbar = plt.colorbar(label='Grey-level difference [0 - 255]', shrink=0.6)
    cbar.ax.tick_params(labelsize=18)
    cbar.set_label('Grey-level difference [0 - 255]', size=20)
    plt.xlabel("$x$-position [pixel]")
    plt.ylabel("$y$-position [pixel]")
    plt.grid(False)
    plt.show()

    collapsed_difference = np.mean(diffim, axis=0)
    x_line = np.arange(diffim.shape[1])

    print(f'\nNumber = {inw}')
    plt.figure(figsize=(8,5))
    plt.plot(x_line,collapsed_difference, color='blue')
    plt.ylabel("Grey-level difference [0-255]")
    plt.xlabel("$x$-position [pixel]")
    plt.grid(True)
    plt.minorticks_on()
    plt.tight_layout(pad=0)
    plt.show()

#### Deformation comparison

In [ ]:
import image_process_tool_box as ipt
import matplotlib.pyplot as plt
import os
import numpy as np

plt.style.use(['science', 'no-latex','ieee', 'grid'])

plt.rcParams.update({
    'font.family': 'Calibri',
    'font.size': 15,
    'axes.labelsize': 15,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
    'lines.linewidth': 1.5,
    'lines.markersize': 3,

    # Border styling
    'axes.linewidth': 1.1,     
    'xtick.major.width': 1.1,    
    'ytick.major.width': 1.1,    
    'xtick.minor.width': 0.7,    
    'ytick.minor.width': 0.7,

    'xtick.major.size': 5,      
    'ytick.major.size': 5,
    'xtick.minor.size': 3,      
    'ytick.minor.size': 3,
})

batch = 11
excel_path = rf"output\excel_docs\excel_{batch}"
excell = r"output\excel_docs\error_separation"



a_metrics, ameas_error, _, _, _ = ipt.read_spec_excel(
    excel_path, doc_num=None)



textures = [
    'none',
    'sinusoidal',
    'cubic',
    'perlin_blobs'
]
tex = 3

corrected_data_folder = ['after_corr_none',
                         'after_corr_cubic',
                         'after_corr_perlin_blobs',
                         'after_corr_sinusoidal'
                         ]

myd_p_metrics, change, _, _, _ = ipt.read_spec_excel(
            excell, 
            doc_num=58+tex, 
            doc_name='my_doc',
            )

# Corrected data
correct_excel = rf"output\excel_docs\{corrected_data_folder[tex]}"
dmetrics, dmeas_error, _, _, _ = ipt.read_spec_excel(
    correct_excel, doc_num=batch, doc_name=f'corrected_batch')

mask = (ameas_error[:,1] != 0) & (ameas_error[:,0] != 0)
mask2 = change[:,3]!=0
mask3 = (dmeas_error[:,0])!=0

# ratio = dmeas_error[mask,0] / ameas_error[mask,0]
print(f'Minimum  RMSE: {np.min(dmeas_error[mask3,0])}')

savef = rf"output\Slices\slice_scatter_plots\{textures[tex]}"
plt.figure(figsize=(5,3),dpi=300)
plt.scatter(myd_p_metrics[mask2,1],change[mask2,3], color='black')
plt.xlabel("MIG")
plt.ylabel('$A_{amp}$')
plt.tight_layout()   
plt.grid('on')
plt.ylim([0,3.5])
plot_save = os.path.join(savef,f'amp_{textures[tex]}.png')
plt.savefig(plot_save)
plt.show()


# plt.scatter(a_metrics[mask,1],dmeas_error[mask,0])
# plt.title("Discrete")
# plt.show()


plt.scatter(a_metrics[mask,1],ameas_error[mask,0])
plt.title("Analy")
plt.show()



In [ ]:
# Random autocorrelation figures
import matplotlib.pyplot as plt
import image_process_tool_box as ipt
import os
import cv2
from scipy import stats

# Plot look settings
plt.style.use(['science', 'no-latex','ieee', 'grid'])

plt.rcParams.update({
    'font.family': 'Calibri',
    'font.size': 14,
    'axes.labelsize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 11,
    'lines.linewidth': 1.25,
    'lines.markersize': 3,

    # Border styling
    'axes.linewidth': 1.0,     
    'xtick.major.width': 1.0,    
    'ytick.major.width': 1.0,    
    'xtick.minor.width': 0.6,    
    'ytick.minor.width': 0.6,

    'xtick.major.size': 5,      
    'ytick.major.size': 5,
    'xtick.minor.size': 3,      
    'ytick.minor.size': 3,
})

# Selected pattern classes
batch = 6
batch2 = 6

# Path to first class
imagepath = rf"C:\Users\General User\nokop\pattern2\data\speckle_pattern_img\reference_im\hold_ref_noised\ref_{batch}"
stringss = ipt.get_image_strings(imagepath)

# Path to second class
imagepath2 = rf"C:\Users\General User\nokop\pattern2\data\speckle_pattern_img\reference_im\hold_ref_noised\ref_{batch2}"
stringss2 = ipt.get_image_strings(imagepath2)

# Load image from first class
prefix1 = 568
# prefix1 = 24
number1 = int(prefix1/2)
image1 = os.path.join(imagepath,stringss[number1])
image_1 = cv2.imread(image1,cv2.IMREAD_GRAYSCALE)
plt.imshow(image_1)
# plt.show()

# Get and show subset from first image
ss_save_1 = r"C:\Users\General User\nokop\pattern2\output\plots\Image_subsets"
fige1 = 'low_rpeak'
save_loc1 = os.path.join(ss_save_1,fige1)
subset_1 = ipt.img_subsets(image_1,350,5)
subset1 = plt.figure(figsize=(2.5,2.5))
plt.imshow(subset_1[0,0,:,:], cmap='gray')
plt.xlabel("$x$ [pixel]")
plt.ylabel("$y$ [pixel]")
plt.grid(False)
plt.savefig(save_loc1)
# plt.show()
# Get figure size
size1 = subset1.get_size_inches()
print('Figure1 size =',size1)

op_metrics, omeas_error, op_param, onans, oindicators = ipt.read_spec_excel(r"C:\Users\General User\nokop\pattern1\output\excel_docs\excel_11", doc_num = None)
# Evaluate first image using metrics
print('Image 1 (fine image) analysis\n-----------------------------------')
msf1 = ipt.MSF(image_1) 
mig1 = ipt.MIG(image_1)
ef1 = ipt.Ef(image_1)
miosd1 = ipt.miosd(image_1)
shan1=ipt.ShannonEnt(image_1)
sssig1 = ipt.globalSSSIG(image_1,33,11)
_,_,R_peak1 = ipt.autocorr(image_1, meth=6, cardinality=5, autype='2d')
# R_peak1 = 0
print(f'MSF = {msf1:.2f}\nMIG = {mig1:.2f}\nEf = {ef1:.2f}\nMIOSD = {miosd1:.2f}\nShannon = {shan1:.2f}\nSSSIG = {sssig1:.2f}\nR_peak = {R_peak1:.2f}')
mag, powr, fx, fy = ipt.compute_fft_and_freq(image_1)
power_area1 = ipt.integrate_power_2d(powr, fx, fy)
print('PSA =',power_area1)

# percentiles
mask = omeas_error[:,0]!=0
percentile_in_sample = stats.percentileofscore(op_metrics[mask,0], msf1, kind='strict')
print('\n\n\n--------',percentile_in_sample)

# Load image from second class
# prefix2 = 278
prefix2 = 602
number2 = int(prefix2/2)
image2 = os.path.join(imagepath2,stringss2[number2])
image_2 = cv2.imread(image2,cv2.IMREAD_GRAYSCALE)
plt.imshow(image_2)
# plt.show()

# Subset from second image
fige2 = 'high_rpeak'
save_loc2 = os.path.join(ss_save_1,fige2)
subset_2 = ipt.img_subsets(image_2,300,5)
subset2 = plt.figure(figsize=(2.5,2.5))
plt.imshow(subset_2[0,0,:,:], cmap='gray')
plt.xlabel("$x$ [pixel]")
plt.ylabel("$y$ [pixel]")
plt.grid(False)
plt.savefig(save_loc2)
# plt.show()
# Get figure size
size2 = subset2.get_size_inches()
print('Figure2 size =',size2)

# Evaluate second image using pattern metrics
print('\nImage 2 analysis\n-----------------------------------')
msf2 = ipt.MSF(image_2) 
mig2 = ipt.MIG(image_2)
ef2 = ipt.Ef(image_2)
miosd2 = ipt.miosd(image_2)
shan2=ipt.ShannonEnt(image_2)
sssig2 = ipt.globalSSSIG(image_2,33,11)
_,_,R_peak2 = ipt.autocorr(image_2, meth=6, cardinality=5, autype='2d')
# R_peak2 = 0
print(f'MSF = {msf2:.2f}\nMIG = {mig2:.2f}\nEf = {ef2:.2f}\nMIOSD = {miosd2:.2f}\nShannon = {shan2:.2f}\nSSSIG = {sssig2:.2f}\nR_peak = {R_peak2:.2f}')
mag, powr, fx, fy = ipt.compute_fft_and_freq(image_2)
power_area = ipt.integrate_power_2d(powr, fx, fy)
print('PSA =',power_area)


# Autocorrelation
plt.figure(figsize=(8,12))
figuress = ipt.super_autocorr(image_1,image_2,dx_min=-50, dx_max=50,meth=6,
                              cardinality = 3, image_1_name = "RCh: linewidth = 6", 
                   image_2_name = "Ch: linewidth = 6", legg = False)
plt.ylim(-0.25,1.05)
plt.legend(
        ['Figure 2.12(a)','Figure 2.12(b)'], 
        loc='upper center', 
        bbox_to_anchor=(0.5, -0.15), 
        ncol=3
        )
plt.savefig(os.path.join(ss_save_1,'autocorr.png'))
plt.show()

In [ ]:
# Random autocorrelation figures
import matplotlib.pyplot as plt
import image_process_tool_box as ipt
import os
import cv2
from scipy import stats

# Plot look settings
plt.style.use(['science', 'no-latex','ieee', 'grid'])

plt.rcParams.update({
    'font.family': 'Calibri',
    'font.size': 14,
    'axes.labelsize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 11,
    'lines.linewidth': 1.25,
    'lines.markersize': 3,

    # Border styling
    'axes.linewidth': 1.0,     
    'xtick.major.width': 1.0,    
    'ytick.major.width': 1.0,    
    'xtick.minor.width': 0.6,    
    'ytick.minor.width': 0.6,

    'xtick.major.size': 5,      
    'ytick.major.size': 5,
    'xtick.minor.size': 3,      
    'ytick.minor.size': 3,
})

# Selected pattern classes
batch = 6
batch2 = 6

# Path to first class
imagepath = rf"C:\Users\General User\nokop\pattern2\data\speckle_pattern_img\reference_im\hold_ref_noised\ref_{batch}"
stringss = ipt.get_image_strings(imagepath)

# Path to second class
imagepath2 = rf"C:\Users\General User\nokop\pattern2\data\speckle_pattern_img\reference_im\hold_ref_noised\ref_{batch2}"
stringss2 = ipt.get_image_strings(imagepath2)

# Load image from first class
prefix1 = 568
# prefix1 = 24
number1 = int(prefix1/2)
image1 = os.path.join(imagepath,stringss[number1])
image_1 = cv2.imread(image1,cv2.IMREAD_GRAYSCALE)
plt.imshow(image_1)
plt.show()

# Get and show subset from first image
ss_save_1 = r"C:\Users\General User\nokop\pattern2\output\plots\Image_subsets"
fige1 = 'low_rpeak'
save_loc1 = os.path.join(ss_save_1,fige1)
subset_1 = ipt.img_subsets(image_1,350,5)
subset1 = plt.figure(figsize=(2.5,2.5))
plt.imshow(subset_1[0,0,:,:], cmap='gray')
plt.xlabel("$x$ [pixel]")
plt.ylabel("$y$ [pixel]")
plt.grid(False)
plt.savefig(save_loc1)
# plt.show()


# Load image from second class
# prefix2 = 278
prefix2 = 602
number2 = int(prefix2/2)
image2 = os.path.join(imagepath2,stringss2[number2])
image_2 = cv2.imread(image2,cv2.IMREAD_GRAYSCALE)
plt.imshow(image_2)
plt.show()

# Subset from second image
fige2 = 'high_rpeak'
save_loc2 = os.path.join(ss_save_1,fige2)
subset_2 = ipt.img_subsets(image_2,300,5)
subset2 = plt.figure(figsize=(2.5,2.5))
plt.imshow(subset_2[0,0,:,:], cmap='gray')
plt.xlabel("$x$ [pixel]")
plt.ylabel("$y$ [pixel]")
plt.grid(False)
plt.savefig(save_loc2)
plt.show()



# Autocorrelation
plt.figure(figsize=(8,12))
figuress = ipt.super_autocorr(image_1,image_2,dx_min=-50, dx_max=50,meth=6,
                              cardinality = 50, image_1_name = "RCh: linewidth = 6", 
                   image_2_name = "Ch: linewidth = 6", legg = False)
plt.ylim(-0.25,1.05)
plt.legend(
        ['Figure 2.12(a)','Figure 2.12(b)'], 
        loc='upper center', 
        bbox_to_anchor=(0.5, -0.15), 
        ncol=3
        )
plt.savefig(os.path.join(ss_save_1,'autocorr.png'))
plt.show()

In [ ]:
# How checkerboard patterns behave. Find a way to extract the behaviour automatically.
# Patterns that have similar block_size to image width ratios have the same RMSEs. Many 
# fall through the cracks and are not analysed. So a 600 sample size could result in only 250 
# analysable patterns and these few will result in 3 to 6 unique RMSE/IQR results
import numpy as np

import matplotlib.pyplot as plt
import image_process_tool_box as ipt

plt.style.use(['science', 'no-latex','ieee', 'grid'])

plt.rcParams.update({
    'font.family': 'Calibri',
    'font.size': 14,
    'axes.labelsize': 14,
    'xtick.labelsize': 11,
    'ytick.labelsize': 10,
    'legend.fontsize': 11,
    'lines.linewidth': 1.25,
    'lines.markersize': 3,

    # Border styling
    'axes.linewidth': 1.0,     
    'xtick.major.width': 1.0,    
    'ytick.major.width': 1.0,    
    'xtick.minor.width': 0.6,    
    'ytick.minor.width': 0.6,

    'xtick.major.size': 5,      
    'ytick.major.size': 5,
    'xtick.minor.size': 3,      
    'ytick.minor.size': 3,
})
cb_excel_path = rf"output\excel_docs"
p_metrics,meas_error,p_param,nans,indicators = ipt.read_spec_excel(
    cb_excel_path,doc_num=186)

p_metrics2,meas_error2,p_param2,nans2,indicators2 = ipt.read_spec_excel(
    cb_excel_path,doc_num=187)

mask = (meas_error[:,0] != 0) & (meas_error[:,0] < 1)
mask2 = (meas_error2[:,0] != 0) & (meas_error2[:,0] < 1)

ratio = 500 // (2 * p_param[mask,1])
ratio2 = 500 // (2 * p_param2[mask2,1])

D = np.round(ratio).astype(int) 
D2 = np.round(ratio2).astype(int)

RMSe = meas_error[mask,0]
RMSe2 = meas_error2[mask2,0]

min = np.min(RMSe)
index = np.where(meas_error[:,0]==min)

print(index[0])


plt.figure(figsize=(4,3))
plt.scatter(D,RMSe, marker='^',  color='#000000', s=12, label='Checkerboard')
plt.scatter(D2,RMSe2, marker='o',  color='#56B4E9', s=12, label='RTG checkerboard')

plt.legend()
plt.xlabel("Line width [pixels]")
plt.ylabel("RMSE [pixel]")
plt.show()




# ss_33_err = [0,0.001577,0,0,0,0.000681,0.001577,0.002717,0.000681,0.0002717,0,0.000681,0,0,0.001577,0.001659]

# ss_67_err = [0.0134,0.00143,0,0.0134,0.0108,0.000656,0.0001434,0,0.000656,0,0,0.000656,0.0134,0,0.001434,0.001342]

# ratio = [24,11,28,23,63,14,11,21,16,22,33,15,23,78,12,13]

# plt.scatter(ratio,ss_33_err,label='subset = 33')
# plt.scatter(ratio,ss_67_err,label='subset = 67')
# plt.legend()
# plt.show()


In [ ]:
import cv2
import image_process_tool_box as ipt

impath = r"data\speckle_pattern_img\reference_im"
image_names = ipt.get_image_strings(impath)

image_n = image_names[16]
stitch_deez = os.path.join(impath,image_n)
image = cv2.imread(stitch_deez, cv2.IMREAD_GRAYSCALE)
print(image_n)

power_of_the_darkside = ipt.uncentered_spec_power(image,plot_type = "semilog",plot = False)




# Subpixel stuff

In [ ]:
import image_process_tool_box as iat
import numpy as np
import os
import matplotlib.pyplot as plt
import scienceplots

sinu_save = r"C:\Users\General User\nokop\pattern2\output\plots\Sinusoidal_bias_error.png"

image_folder = r"data\speckle_pattern_img\reference_im"
imlist = iat.get_image_strings(image_folder)

x = np.linspace(0,1,100)

plt.style.use(['science', 'no-latex','ieee', 'grid'])

plt.rcParams.update({
    'font.family': 'Calibri',
    'font.size': 16,
    'axes.labelsize': 16,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 11,
    'lines.linewidth': 1.25,
    'lines.markersize': 3,

    # Border styling
    'axes.linewidth': 1.0,     
    'xtick.major.width': 1.0,    
    'ytick.major.width': 1.0,    
    'xtick.minor.width': 0.6,    
    'ytick.minor.width': 0.6,

    'xtick.major.size': 5,      
    'ytick.major.size': 5,
    'xtick.minor.size': 3,      
    'ytick.minor.size': 3,
})


plt.figure(figsize=(4,3.5))

markers = ['o', 's', 'D', '^', 'v', '<', '>', '*', 'x', '+']

for i, fname in enumerate(imlist):


    image_path = os.path.join(image_folder, fname)
    img = iat.readImage(image_path)
    # plt.imshow(img,cmap='grey')
    # plt.grid(False)
    # plt.axis('off')
    # plt.savefig(
    #     os.path.join(r"C:\Users\General User\nokop\pattern2\data\speckle_pattern_img\reference_im\theory_validate4", f'image{2*i+6}.png'))
    # plt.close()
    # Interpolation bias
    c_ib = iat.compute_bias_prediction(img, plot_spectrum=False, 
                                        algorithm='ic-gn')
    
    # Noise-induced bias
    c_nb = 0
    u_e = (c_ib + c_nb) * np.sin(2 * np.pi * x)

    marker = markers[i % len(markers)]
    plt.plot( x, u_e, label=f"Pattern {i+4}")

# plt.title(f"Bias Error Predictions for {len(imlist)} Images")
plt.ylabel("$e_{sys}$ [pixel]")
plt.xlabel("$u_0$ [pixel]")
# plt.ylim(-0.005, 0.005)
plt.xticks()  
plt.yticks() 
plt.grid(True)
plt.legend(loc='upper center', 
        bbox_to_anchor=(0.5, -0.25), 
        ncol=3
        )
plt.tight_layout()
plt.savefig(sinu_save)
plt.show()

#### Interpolation bias kernel

In [ ]:
# Plotting the interpolation bias kernel
import matplotlib.pyplot as plt
plt.style.use(['science', 'no-latex','ieee', 'grid'])

plt.rcParams.update({
    'font.family': 'Calibri',
    'font.size': 12,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 11,
    'lines.linewidth': 1.25,
    'lines.markersize': 3,

    # Border styling
    'axes.linewidth': 1.0,     
    'xtick.major.width': 1.0,    
    'ytick.major.width': 1.0,    
    'xtick.minor.width': 0.6,    
    'ytick.minor.width': 0.6,

    'xtick.major.size': 5,      
    'ytick.major.size': 5,
    'xtick.minor.size': 3,      
    'ytick.minor.size': 3,
})

eb_save = r"C:\Users\General User\nokop\pattern2\output\plots\Interp_Kern.png"

def cube_transfer(nu):
    """Cubic B-spline transfer function (from MATLAB code)"""
    return 3 * (np.sinc(nu))**4 / (2 + np.cos(2*np.pi*nu))

nu = np.linspace(0,0.5,100)

# 1D bias kernel
def bias(nu):
    nu_min = nu-1
    nu_plus = nu+1
    kerna = nu_min*cube_transfer(nu_min) - nu_plus*cube_transfer(nu_plus)+cube_transfer(nu)*cube_transfer(nu_plus) + cube_transfer(nu)*cube_transfer(nu_min)
    return kerna

Eb = bias(nu)

plt.plot(nu,Eb)
plt.xlabel("Frequency [cycles/pixel]")
plt.ylabel("Interpolation bias kernel")

plt.savefig(eb_save)
plt.show()

#### Slice comparison

In [ ]:

# Line of best fit to get error propogation gradient

import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from scipy import stats
import scienceplots

plt.style.use(['science', 'no-latex','ieee', 'grid'])

plt.rcParams.update({
    'font.family': 'Calibri',
    'font.size': 16,
    'axes.labelsize': 16,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 11,
    'lines.linewidth': 1.25,
    'lines.markersize': 3,

    # Border styling
    'axes.linewidth': 1.0,     
    'xtick.major.width': 1.0,    
    'ytick.major.width': 1.0,    
    'xtick.minor.width': 0.6,    
    'ytick.minor.width': 0.6,

    'xtick.major.size': 5,      
    'ytick.major.size': 5,
    'xtick.minor.size': 3,      
    'ytick.minor.size': 3,
})

paths_dictionary = {
    1:  r"output\Slices\slice_binaries\4mm_go",
    2:  r"output\Slices\slice_binaries\4mm",
    3:  r"output\Slices\slice_binaries\correct2"
}

folder_number = 3
slice_bin = paths_dictionary.get(folder_number)

string = 'X_disp'

prefix_range = [0,2,4]

# load and plot all files\
plt.figure(figsize=(4,3))

for file_1 in prefix_range:

    file_1_path2 = os.path.join(slice_bin, f'{file_1}_slice_{string}.npy')
    var = np.load(file_1_path2)

    y = var[:,0]
    x = var[:,1]
    u_0 = var[:,2]
    u_m = var[:,3]

    print(f"x shape = {x.shape}")
    print(f"u0 = {u_0.shape}")


    # Linear
    m, b,_,_,_ = stats.linregress(u_0,y)

    print(f"Slope (m): {m}")
    print(f"Y - intercept (b): {b}")

    # Figure with line
    y_line = m * u_0 + b


    # plt.plot(x,y,label = f"Error")
    plt.plot(u_0,y,label = f"Error")

    # plt.plot(u_0,y_line, label = f'Best fit - slope: {m:.4f}')
    plt.legend(
        ['Pattern 4','Pattern 5','Pattern 6'], 
        loc='upper center', 
        bbox_to_anchor=(0.5, -0.25), 
        ncol=3
        )
    # plt.title("Error at different edge displacements")
plt.ylabel("$e_{sys,i}$ [pixel]")
plt.xlabel("$u_0$ [pixel]")
plt.ylim(-0.005, 0.005)
plt.grid(True)
save_edgre = os.path.join(r"C:\Users\General User\nokop\pattern2\data\speckle_pattern_img\reference_im\theory_validate4",
                        f'validate3.png')
plt.savefig(save_edgre)
plt.show()

# 2. Image subsets

#### Pattern parameter bounds identification

In [ ]:
# Get subsets
import image_process_tool_box as ipt
import os
import matplotlib.pyplot as plt

subset_save_path = r"output\image_matrix"

read_path = [
    rf"C:\Users\General User\nokop\Hold\Discussion\12_batches_of_500_gridbased_deform\Images\reference\ref_{batch}" 
    for batch in range(12)
]

for batch_number in range(len(read_path)):

    if batch_number != 4:
        continue

    # Make a save location for subsets
    subset_save = os.path.join(subset_save_path, f"batch_{batch_number}")
    if not os.path.exists(subset_save):
        os.makedirs(subset_save)

    print(f"---------------\nBatch {batch_number}")
    indx_1 = 0
    indx_2 = indx_1 + 119

    # get the names of the images
    img_names = ipt.get_image_strings(read_path[batch_number])

    pathe_to_img1 = os.path.join(read_path[batch_number], img_names[indx_1])
    pathe_to_img2 = os.path.join(read_path[batch_number], img_names[indx_2])

    image1 = ipt.readImage(pathe_to_img1)
    image2 = ipt.readImage(pathe_to_img2)

    plt.imshow(image1, cmap='grey')
    plt.axis('off')
    plt.show()

    plt.imshow(image2, cmap='grey')
    plt.axis('off')
    plt.show()

    # Image subsets
    img1_subsets = ipt.img_subsets(image1, 250, 5)
    plt.imshow(img1_subsets[0,0,:,:], cmap='grey')
    plt.axis('off')
    save_sub = os.path.join(subset_save_path, f"b{batch_number}_subset_1.png")
    plt.savefig(save_sub, bbox_inches='tight', pad_inches=0, dpi=300)
    plt.show()

    img2_subsets = ipt.img_subsets(image2, 250, 5)
    plt.imshow(img2_subsets[0,0,:,:], cmap='grey')
    plt.axis('off')
    save_sub = os.path.join(subset_save_path, f"b{batch_number}_subset_2.png")
    plt.savefig(save_sub, bbox_inches='tight', pad_inches=0, dpi=300)
    plt.show()



#### Perlin Noise Tutorial

In [ ]:
import noise

# 2D perlin noise takes an x and y value and returns one value.
# Entirely contuous so it can be defined deformed without pixel interpolation
# artifacts

x = 0.001
y = 0.001
perlin_value = noise.pnoise2(x, y)
print(perlin_value)


In [ ]:
'''
We will use the indices of the array - i.e., the row and column numbers - as the inputs for pnoise2
Basically use the integer "coordinates" of each pixel.

'''

def gen_perlin_2d(n_x, n_y, scale=1, **kwargs):
  
  # Create an array for holding Perlin noise
  # y coordinates correspond to the rows
  # Generate a n_x by n_y "empty image"
  perlin_data = np.zeros((n_y, n_x))
  
  # Largest dimension - use to scale values
  max_n=max(n_y, n_x)

  # Compute noise using indices of the array
  for i in range(n_y): 
    for j in range(n_x):
      perlin_data[i,j] = noise.pnoise2(
        j/max_n * scale, # pass x value first (j)
        i/max_n * scale,
        **kwargs # Additional keyword arguments to pass to pnoise2
      )
      
  return perlin_data

'''
You might have noticed that I didn't pass the array's indices, `i` and `j`, 
directly to `pnoise2`—instead, I adjusted them using the array’s size and 
the `scale` parameter.
'''

def plot_perlin_art(perlin_data, show_plot=True):
  # Plot the data as a heatmap (each value = different colour)
  fig, ax = plt.subplots(1, 1)
  ax.imshow(perlin_data)
  ax.axis("off") # remove axes
  ax.set_aspect("equal") # equal aspect ratio
  if show_plot:
    plt.show()
  
  # Return figure and axes handles
  return fig, ax


# Define dimensions of the data
n_x = 1000
n_y = 1000

# Generate data
perlin_data = gen_perlin_2d(n_x, n_y, scale=25)

# Plot
fig, ax = plot_perlin_art(perlin_data)


In [ ]:
# Original - smooth
perlin_data = gen_perlin_2d(n_x=1000, n_y=1000)
fig, ax = plot_perlin_art(perlin_data)

# Decreasing resolution
perlin_data = gen_perlin_2d(n_x=50, n_y=50)
fig, ax = plot_perlin_art(perlin_data)

# Even more pixelated
perlin_data = gen_perlin_2d(n_x=10, n_y=10)
fig, ax = plot_perlin_art(perlin_data)

In [ ]:
'''
If we want to change the magnitude of the values passed to pnoise2, we can use the function’s scale argument.
This argument lets us create larger or smaller patterns:
'''

n_x = 1000
n_y = 1000

# Larger pattern with scale < 1
perlin_data = gen_perlin_2d(n_x, n_y, scale=0.1)
fig, ax = plot_perlin_art(perlin_data)

# Default scale (1)
perlin_data = gen_perlin_2d(n_x, n_y, scale=1)
fig, ax = plot_perlin_art(perlin_data)

# Smaller patterns with scale > 1
perlin_data = gen_perlin_2d(n_x, n_y, scale=10)
fig, ax = plot_perlin_art(perlin_data)

#### Gaussian noise pattern and texture

In [ ]:
import numpy as np
from PIL import Image, ImageFilter

def create_noise_texture(width, height, sigma, filename):
    """
    Generates and saves a noise texture.

    Args:
        width (int): Width of the texture.
        height (int): Height of the texture.
        sigma (int): Standard deviation of the Gaussian noise.
        filename (str): Name of the file to save the texture.
    """
    noise = Image.effect_noise((width, height), sigma)
    noise = noise.filter(ImageFilter.GaussianBlur(radius=1))
    noise.save(filename)
    return noise

# Example usage
width, height = 2000, 500
subtle_texture = create_noise_texture(width, height, 25, "subtle_noise.png")
medium_texture = create_noise_texture(width, height, 75, "medium_noise.png")
strong_texture = create_noise_texture(width, height, 150, "strong_noise.png")

strong_texture.show()

#### CLAHE image

In [ ]:
# CLAHE histogram equalisation
# https://www.geeksforgeeks.org/clahe-histogram-eqalization-opencv/
# https://www.youtube.com/watch?v=tn2kmbUVK50

import cv2
import numpy as np
from IPython.display import clear_output
import os
import image_process_tool_box as ipt


# image_folder = r'C:\Users\nokop\OneDrive - Stellenbosch University\STELLENBOSCH\08 Masters\speckle_pattern_evaluation\data\speckle_pattern_img\Miscelleneous'
image_folder = r'data\speckle_pattern_img\reference_im'
# Get the image files. Reads files in the specified folder that end with .tif,
# have a number at the start of the file name and the number is even. It sorts 
# the files numerically based on the number at the start of the file name.

im_typ = 'tif'

image_files = sorted(
    [f for f in os.listdir(image_folder) 
     if f.endswith(f".{im_typ}") and f.split('_')[0].isdigit() and int(f.split('_')[0]) % 2 == 0],
    key=lambda x: int(x.split('_')[0])
)

print(len(image_files))


# Start numbering from 30 and increment by 2 for each image
save_number = 48

limit = len(image_files)
limit = 3
file_number = 0




def clahe_image(image):

    """ 
    Function performs histogram equalisation on an image. 
    The new image can overrite the old image (but this is not advised)
    Creates a figure showing the image differences and their associated histograms.
    
    """
    if image.ndim > 2:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Apply transformations
    clahe = cv2.createCLAHE(clipLimit=5)
    final_img = clahe.apply(image) + 30

    # Figure
    fig = plt.figure(figsize=(12, 10))

    # Original image and histogram
    plt.subplot(2, 2, 1)
    plt.imshow(image, cmap='gray')
    plt.title('Original Image')
    plt.axis('off')

    plt.subplot(2, 2, 2)
    plt.hist(image.ravel(), 256, [0, 256])
    plt.title('Original Histogram')
    plt.xlim([0, 256])

    # CLAHE image and histogram
    plt.subplot(2, 2, 3)
    plt.imshow(final_img, cmap='gray')
    plt.title('CLAHE Image')
    plt.axis('off')

    plt.subplot(2, 2, 4)
    plt.hist(final_img.ravel(), 256, [0, 256])
    plt.title('CLAHE Histogram')
    plt.xlim([0, 256])

    plt.tight_layout()

    return final_img, fig

clahe_path = os.path.join(image_folder, 'Clahe')
if not os.path.exists(clahe_path):
    os.makedirs(clahe_path)

for file_number in range(len(image_files)):
    image_path = os.path.join(image_folder, image_files[file_number])
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    clahe, figurer = clahe_image(image)
    image_save = os.path.join(clahe_path, f'{file_number*2}_levelised.tif')
    cv2.imwrite(image_save,clahe)
    print(f'Image {file_number*2} saved.')



# plt.show()
# # Save image

# # Compare subsets
# plt.subplot(1,2,1)
# plt.imshow(ipt.img_subsets(image,100,5)[0,0,:,:], cmap='gray')
# plt.title("Original image")
# plt.subplot(1,2,2)
# plt.imshow(ipt.img_subsets(clahe,100,5)[0,0,:,:], cmap='gray')
# plt.title("Clahe image")
# plt.show()


# Invert and threshold (Otsu) images 


In [ ]:
import image_process_tool_box as iat 

ref_image_folder = r"C:\Users\General User\nokop\pattern2\data\speckle_pattern_img\reference_im"


img_optm = r'C:\Users\General User\nokop\pattern2\data\speckle_pattern_img\reference_im\temp'
deformed_folder = r"C:\Users\General User\nokop\pattern2\data\speckle_pattern_img\deformed_im\temp"

bin_invert = r"C:\Users\General User\nokop\pattern2\data\speckle_pattern_img\deformed_im\Bin_invert_perlin"

import time

# New prefixes
# iat.ordered_prefix(ref_image_folder,refer='ref',start_at=28)
# iat.rename_img(random_folder,  parity='even')     

# time.sleep(3)

# Add Gaussian blur
# iat.random_pixel_background_noise(ref_image_folder, min_val=75, max_val=150, image_type='tif', name_change='no', par='even')
# time.sleep(2.5)
# iat.gaussian_blur_images(
#     ref_image_folder,
#     size=5,
#     sig_y=1.0,
#     par='even'
# )

# iat.gaussian_noise_images(
#     ref_image_folder,  
#     par = 'even')

# threshold
# iat.imthresh(ref_image_folder,ref_image_folder,refer='ref', keep_name=True)

# Invertf
# iat.iminvert(random_folder,bin_invert,refer='def')

# Rename deformed images
# iat.rename_img(ref_image_folder)
# iat.im_resize(img_optm,2000,500)

# # Make Turing
# t = [6,7,8,9,10,11]

# for clsss in t:
        
#     random_folder = rf"data\speckle_pattern_img\reference_im\ref_{clsss}"
#     iat.make_turing(random_folder,rep=25, radius=1, 
#                     sharpen_percent=250, size=None, 
#                     replace=False, par='even')
    


# iat.make_turing(img_optm,rep=40, radius=1, sharpen_percent=250, size=None, replace=True, par='even')

# Move files to a different folder
# iat.move_files(random_folder,ref_image_folder,suffix='Generated_spec_image')


In [ ]:
import numpy as np

images = 20
prefixes = np.arange(0,40,2)
print(f"Expected prefixes: {prefixes}\n")

itran = images / 2

for i in range(int(itran)):
    anal = i * 2
    grid = anal + itran * 2

    print(f"Analytical prefix = {anal}")
    print(f"Grid-based prefix = {grid}")



In [ ]:

import cv2
import numpy as np

# Read the image
image = cv2.imread(r"C:\Users\General User\nokop\pattern2\data\speckle_pattern_img\reference_im\0_Generated_spec_image.tif")

# Remove one row (last row)
new_image = image[:-1, :]

# Save the result
cv2.imwrite(r"C:\Users\General User\nokop\pattern2\data\speckle_pattern_img\reference_im\0_Generated_spec_image.tif", new_image)

print(f"Original shape: {image.shape}")
print(f"New shape: {new_image.shape}")

# Some error curves

In [ ]:
# Line of best fit to get error propogation gradient

import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from scipy import stats


plt.style.use(['science', 'no-latex','ieee', 'grid'])

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 5,
    'axes.labelsize': 5,
    'xtick.labelsize': 5,
    'ytick.labelsize': 5,
    'legend.fontsize': 5,
    'lines.linewidth': 1.25,
    'lines.markersize': 3
})

path_to_slice_bin2 = r"output\Slices\slice_binaries\2mm_rel"
path_to_slice_bin4 = r"output\Slices\slice_binaries\4mm_rel"
path_to_slice_bin6 = r"output\Slices\slice_binaries\6mm_rel"
path_to_slice_bin8 = r"output\Slices\slice_binaries\8mm_rel"

file_1 = 2
string = 'X_disp'

file_1_path2 = os.path.join(path_to_slice_bin2, f'{file_1}_slice_{string}.npy')
load_file2 = np.load(file_1_path2)
file_1_path4 = os.path.join(path_to_slice_bin4, f'{file_1}_slice_{string}.npy')
load_file4 = np.load(file_1_path4)
file_1_path6 = os.path.join(path_to_slice_bin6, f'{file_1}_slice_{string}.npy')
load_file6 = np.load(file_1_path6)
file_1_path8 = os.path.join(path_to_slice_bin8, f'{file_1}_slice_{string}.npy')
load_file8 = np.load(file_1_path8)


columns = 1

y2 = load_file2[:,0]
x2 = load_file2[:,columns]
y4 = load_file4[:,0]
x4 = load_file4[:,columns]
y6 = load_file6[:,0]
x6 = load_file6[:,columns]
y8 = load_file8[:,0]
x8 = load_file8[:,columns]


plt.figure()
plt.plot(x2,y2,label = f"2mm edge deformation")
plt.plot(x4,y4,label = f"4mm")
plt.plot(x6,y6,label = f"6mm")
plt.plot(x8,y8,label = f"8mm")
# plt.plot(x,y_line,label = f'Best fit: slope {(10**6)*m:.3f} e^-6')
plt.legend()
# plt.title("Relative error")
plt.ylabel("y-Direction mean error [pixels]")
plt.xlabel("Location on image [pixels]")
plt.ylim(-0.005, 0.0055)
plt.grid(True)
save_edgre = os.path.join(r"C:\Users\General User\nokop\pattern2\output\plots", 
                          f'Relative_edge_behaviour_relative.png')
plt.savefig(save_edgre)
plt.show()




In [ ]:
# Line of best fit to get error propogation gradient

import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from scipy import stats
import SciencePlots 


plt.style.use(['science', 'no-latex','ieee', 'grid'])

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 5,
    'axes.labelsize': 5,
    'xtick.labelsize': 5,
    'ytick.labelsize': 5,
    'legend.fontsize': 5,
    'lines.linewidth': 1.25,
    'lines.markersize': 3
})

path_to_slice_bin2 = r"output\Slices\slice_binaries\2mm"
path_to_slice_bin4 = r"output\Slices\slice_binaries\4mm"
path_to_slice_bin6 = r"output\Slices\slice_binaries\6mm"
path_to_slice_bin8 = r"output\Slices\slice_binaries\8mm"

file_1 = 2
string = 'X_disp'

file_1_path2 = os.path.join(path_to_slice_bin2, f'{file_1}_slice_{string}.npy')
load_file2 = np.load(file_1_path2)
file_1_path4 = os.path.join(path_to_slice_bin4, f'{file_1}_slice_{string}.npy')
load_file4 = np.load(file_1_path4)
file_1_path6 = os.path.join(path_to_slice_bin6, f'{file_1}_slice_{string}.npy')
load_file6 = np.load(file_1_path6)
file_1_path8 = os.path.join(path_to_slice_bin8, f'{file_1}_slice_{string}.npy')
load_file8 = np.load(file_1_path8)

columns = 1

y2 = load_file2[:,0]
x2 = load_file2[:,columns]
y4 = load_file4[:,0]
x4 = load_file4[:,columns]
y6 = load_file6[:,0]
x6 = load_file6[:,columns]
y8 = load_file8[:,0]
x8 = load_file8[:,columns]

# Linear
m, b,_,_,_ = stats.linregress(x,y)

print(f"Slope (m): {m}")
print(f"Y - intercept (b): {b}")

# Figure with line
y_line = m * x + b

plt.figure()
plt.plot(x2,y2,label = f"2mm edge deformation")
plt.plot(x4,y4,label = f"4mm")
plt.plot(x6,y6,label = f"6mm")
plt.plot(x8,y8,label = f"8mm")
# plt.plot(x,y_line,label = f'Best fit: slope {(10**6)*m:.3f} e^-6')
plt.legend()
# plt.title("Absolute error")
plt.ylabel("y-Direction mean error [pixels]")
plt.xlabel("Location on image [pixels]")
plt.ylim(-0.005, 0.035)
plt.grid(True)
save_edgre = os.path.join(r"C:\Users\General User\nokop\pattern2\output\plots", 
                          f'all_edge_behaviour.png')
plt.savefig(save_edgre)
plt.show()


In [ ]:
import numpy as np
import sundic.post_process as sdpp
from scipy.interpolate import griddata, RBFInterpolator
from pyNastran.bdf.bdf import BDF
from pyNastran.op2.op2 import OP2
import matplotlib.pyplot as plt



# region Functions
def smooth_field(matrix, source_points, destination_points, kern):

    '''
    This functioin uses radial basis function interpolation to generate a smooth displacement field based
    on the source and destination points which are mapped to a grid of image dimensions. The source and destination
    points must be aligned as close as possible to the image.

    The matrix is for alignment purposes only. The source and destination points are used to generate the displacement field.
    The shape of the matrix must be of the following form: (h, w, 3) where h and w are the image dimensions.
    '''
    h, w = matrix.shape[:2]  

    # Create a grid of coordinates (destination grid). The grid size is based on the image dimensions.
    # The grid will be used to obtain the remap coordinates for the deformed image.
    grid_x, grid_y = np.meshgrid(np.arange(w), np.arange(h))
    grid_x_flatten = grid_x.flatten()
    grid_y_flatten = grid_y.flatten()

    ''' 
    The type of RBF that is used to interpolate the displacement field is described by the `kernel` parameter.
    The following kernels are available:
    Kernels: thin_plate_spline 
            linear
            cubic
            quintic

    `epsilon` must be specified if `kernel` is not one of {'quintic', 'cubic', 'thin_plate_spline', 'linear'}.
    Shape parameter that scales the input to the RBF. If `kernel` is
        'linear', 'thin_plate_spline', 'cubic', or 'quintic', this defaults to
        1 and can be ignored because it has the same effect as scaling the
        smoothing parameter. Otherwise, this must be specified.   
    The kernel parameter is selected through the last argument of the function.       
    '''
    kernel_options = {1: 'thin_plate_spline', 2: 'linear', 3: 'cubic', 4: 'quintic'}

    print("Generating displacement field.  Running TPS interpolation with kernel: ", kernel_options.get(kern))

    kerne = kernel_options.get(kern)
    if kerne is None:
        raise ValueError("Invalid kernel option. Use 1 for thin_plate_spline, 2 for linear, 3 for cubic, or 4 for quintic.")


    rbf_interpolator_x = RBFInterpolator(source_points, destination_points[:, 0] - source_points[:, 0], kernel=kerne)
    rbf_interpolator_y = RBFInterpolator(source_points, destination_points[:, 1] - source_points[:, 1], kernel=kerne)
    '''
    Apply RBF interpolation to get a smooth displacement field. The displacement field x and y components are 
    reshaped to the image dimensions by stacking the grid coordinates and then reshaping the result. The displacement
    field is interpolated at the grid coordinates.
    '''
    # Sample at pixel coordinates. 
    dx = rbf_interpolator_x(np.column_stack([grid_x_flatten, grid_y_flatten])).reshape(h, w)
    dy = rbf_interpolator_y(np.column_stack([grid_x_flatten, grid_y_flatten])).reshape(h, w)

    return dx, dy, rbf_interpolator_x, rbf_interpolator_y


def load_fe_nodes(bdf_path, op2_path):

    """
    Loads FE data and scales it to an image. The aspect ratios of the image and the 
    FE data must be identical. Returns scaled vectors

    Uses class-based method of reading the respective files.

    Unscaled
    """
    model = OP2()
    model.read_op2(op2_path)
    bdf = BDF()
    bdf.read_bdf(bdf_path)

    # Extract the displacement data from the op2 file. Assuming isubcase = 1 and static analysis
    itime = 0
    isubcase = 1
    disp = model.displacements[isubcase]
    # Extract the translational displacements
    txyz = disp.data[itime, :, :3]
    nodes = np.array([bdf.nodes[nid].get_position() for nid in bdf.nodes])

    # Extract only the x and y components for 2D visualization (2d dic)
    nodes_2d = nodes[:, :2]
    displacements_2d = txyz[:, :2]

    deformed_nodes_2d = nodes_2d + displacements_2d

    return nodes_2d, deformed_nodes_2d

# endregion


image_height = 500
image_width = 2000
x_scale = 1000

# Quick error
# 1. Load DIC data
sundic_data_path = r"C:\Users\General User\nokop\pattern2\output\sundic_bin\250_results.sdic"
sundic_data, nRows, nCols = sdpp.getDisplacements(
                sundic_data_path,
                -1,
                smoothWindow=25
            )

x_coord, y_coord = sundic_data[:, 0], sundic_data[:, 1]
X_disp, Y_disp = sundic_data[:, 3], sundic_data[:, 4]
dic_magnitude = np.sqrt(X_disp**2 + Y_disp**2)
sundic_points = np.column_stack((x_coord,y_coord))          # For interpolation


# 2. Load FE data
op2_path = r"data\FEA\All_flats\Flat_30mm_mesh\Flat30_4mm_as_defined_dsg00.op2"
bdf_path = r"data\FEA\All_flats\Flat_30mm_mesh\Flat30_4mm_as_defined.bdf"

nodes_2d, deformed_nodes_2d = load_fe_nodes(bdf_path,op2_path)

displacements_2d = deformed_nodes_2d - nodes_2d
fem_xcoord = nodes_2d[:, 0] * x_scale
fem_ycoord = nodes_2d[:, 1] * x_scale
fem_x_disp = displacements_2d[:, 0] * x_scale  
fem_y_disp = displacements_2d[:, 1] * x_scale

fem_magnitude = np.sqrt(fem_x_disp**2 + fem_y_disp**2)


dx, dy, FEx_rbf_interp, FEy_rbf_interp = smooth_field(
            np.zeros((image_height, image_width)),
            nodes_2d,
            deformed_nodes_2d,
            3
        )

# 3. Corrected FE coordinate system (converted to Lagrangian coordinate system)
lag_dx = FEx_rbf_interp(np.column_stack([fem_xcoord, fem_ycoord]))
lag_dy = FEy_rbf_interp(np.column_stack([fem_xcoord, fem_ycoord]))

fem_xcoord_lag = fem_xcoord - lag_dx
fem_ycoord_lag = fem_ycoord - lag_dy

fem_points = np.column_stack((fem_xcoord_lag, fem_ycoord_lag)) 

# 4. Select which data to compare (X displacements, y displacements or displacement magnitude)
value = 0
if value == 0:
    fem_value = fem_x_disp
    dic_value = X_disp
    string = 'X_displacement'
elif value == 1:
    fem_value = fem_y_disp
    dic_value = Y_disp
    string = 'Y_disp'
elif value == 2:
    fem_value = fem_magnitude
    dic_value = dic_magnitude
    string = 'Magnitude'
else:
    raise ValueError("Invalid value")

# 5. Interpolation and comparison
interpolated_FEM_values = griddata(fem_points, 
                                fem_value, 
                                sundic_points, 
                                method='cubic'
                                )


# 6. Calculate the error
# Reshape data
dic_value = dic_value.reshape(nCols,nRows)
interpolated_FEM_values = interpolated_FEM_values.reshape(nCols,nRows)

errors = (dic_value - interpolated_FEM_values)


# 7. Show errors
x_grid = x_coord.reshape(nCols, nRows)          # For display purposes
plt.figure(figsize=(10, 8))
plt.imshow(errors.T, cmap='jet', interpolation='none', 
                extent=(x_coord.min(), x_coord.max(), y_coord.min(), y_coord.max()), 
                origin='lower')
plt.colorbar(label=f'Error {string}')
plt.title(f'{string} Error Distribution')
plt.xlabel('X')
plt.ylabel('Y')
plt.gca().invert_yaxis()
plt.show()

# 8. Get bias, std. error and RMSE
MBE_d2f = np.nanmean(errors)
SDE_d2f = np.std(errors)
RMSE_d2f = np.sqrt(MBE_d2f**2 + SDE_d2f**2)

print("bias:",MBE_d2f )
print("std. err:",SDE_d2f )
print("RMSE:",RMSE_d2f )


x_grid = x_coord.reshape(nCols, nRows)
collapsed_errorgrid = np.nanmean(errors, axis=1)
x_line = np.mean(x_grid, axis=1)
plt.style.use(['science', 'no-latex','ieee', 'grid'])

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 5,
    'axes.labelsize': 5,
    'xtick.labelsize': 5,
    'ytick.labelsize': 5,
    'legend.fontsize': 5,
    'lines.linewidth': 1.0,
    'lines.markersize': 1.5
})
plt.figure(figsize=(10, 6))

# Mask the x domain 
x_start_slice = 0
x_line_mask = x_line >= x_start_slice

# Plot slice with x-domain mask
plt.plot(x_line[x_line_mask] ,collapsed_errorgrid[x_line_mask], color='black', linewidth=1.5)
plt.xlabel("Pixels")
plt.ylabel(f"y-averaged error {string}")
plt.ylim(-0.0015, 0.0015)
plt.grid(True)
plt.show()

# Parameter-space exploration

In [ ]:

# Perlin noise

import numpy as np
import matplotlib.pyplot as plt
import image_process_tool_box as ipt
# plt.style.use(['science', 'no-latex','ieee', 'grid'])

# plt.rcParams.update({
#     'font.family': 'DejaVu Sans',
#     'font.size': 5,
#     'axes.labelsize': 5,
#     'xtick.labelsize': 5,
#     'ytick.labelsize': 5,
#     'legend.fontsize': 5,
#     'lines.linewidth': 1.0,
#     'lines.markersize': 1.5
# })
# Configuration
config = {
    'octave': 5,
    'pers': 0.01,
    'lacu': 0.9,
    'textu': 'none',
    'num_points': 30
}

def generate_mig_curve(min_scale, max_scale, show_images=False):

    # Doc string
    """Generate MIG values for a scale range and optionally show images"""

    # List for storing parameter values, corresponding metric values and generated images
    scales = np.exp(np.linspace(np.log(min_scale), np.log(max_scale), config['num_points']))
    mig_values = []
    images = []
    
    for i, scale in enumerate(scales):

        img = ipt.generate_single_perlin_image(500, 500, 
                                             scale=scale,
                                             octaves=config['octave'],
                                             persistence=config['pers'],
                                             lacunarity=config['lacu'],
                                             texture_function=config['textu'])
        
        mig = ipt.ShannonEnt(img)
        mig_values.append(mig)
        
        if show_images and (i == 0 or i == len(scales)-1):
            images.append((scale, img, mig))
        
        # print(f'Scale: {scale:.4f}, MIG: {mig:.4f}')
    
    return scales, mig_values, images

# Your custom scale range (edit these values)
my_min_scale = 0.0001
my_max_scale = 0.45
# Generate and plot both curves
test_range_minimum = 0.0001
test_range_maximum = 150

base_scales, base_mig, _ = generate_mig_curve(test_range_minimum, test_range_maximum)
my_scales, my_mig, my_images = generate_mig_curve(my_min_scale, my_max_scale, show_images=True)

# Plot comparison
parr = 'Scale'
ylabb = 'MIG'
xlabb = f'{parr}'

plt.figure(figsize=(12, 6))
plt.semilogx(base_scales[:-1], base_mig[:-1], 'b-', label=f'Test range ({test_range_minimum}-{test_range_maximum})')
plt.semilogx(my_scales, my_mig, 'ro-', label=f'Active range ({my_min_scale}-{my_max_scale})')
plt.xlabel(xlabb)
plt.ylabel(ylabb)
plt.grid(True, which="both", ls="--")
plt.legend()
plt.show()

# Show your first and last images
for scale, img, mig in my_images:
    plt.figure()
    plt.imshow(img, cmap='gray')
    plt.title(f'Your Scale: {scale:.4f}\nMIG: {mig:.4f}')
    plt.axis('off')
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import image_process_tool_box as ipt
import cv2

plt.style.use(['science', 'no-latex','ieee', 'grid'])

plt.rcParams.update({
    'font.family': 'Calibri',
    'font.size': 12,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 11,
    'lines.linewidth': 1.25,
    'lines.markersize': 3,
    'axes.linewidth': 1.0,     
    'xtick.major.width': 1.0,    
    'ytick.major.width': 1.0,    
    'xtick.minor.width': 0.6,    
    'ytick.minor.width': 0.6,
    'xtick.major.size': 5,      
    'ytick.major.size': 5,
    'xtick.minor.size': 3,      
    'ytick.minor.size': 3,
})

def study_turing_parameter(param_name, test_min, test_max, selected_min, selected_max):
    """Study Turing transformation parameters (radius or rep)"""
    
    # Generate base pattern once
    base_pattern = ipt.generate_single_perlin_image(
        250, 250, 
        scale=0.05, 
        octaves=5, 
        persistence=0.5, 
        lacunarity=2.0, 
        texture_function='sinusoidal'
    )
    cv2.imwrite('temp_base.tif', base_pattern)
    
    # Default Turing config
    defaults = {'rep': 25, 'radius': 5, 'sharpen_percent': 250}
    
    # Generate test range values
    test_values = np.linspace(test_min, test_max, 50)
    selected_values = np.linspace(selected_min, selected_max, 20)
    
    # Calculate MIG and Shannon for test range
    test_mig = []
    test_shannon = []
    
    for val in test_values:
        params = defaults.copy()
        params[param_name] = int(val)
        
        img_pil = ipt.make_turing_single('temp_base.tif', **params)
        img = np.array(img_pil.getchannel(0))
        
        mig = ipt.MIG(img)
        shannon = ipt.ShannonEnt(img)
        test_mig.append(mig)
        test_shannon.append(shannon)
    
    # Calculate MIG and Shannon for selected range
    selected_mig = []
    selected_shannon = []
    
    for val in selected_values:
        params = defaults.copy()
        params[param_name] = int(val)
        
        img_pil = ipt.make_turing_single('temp_base.tif', **params)
        img = np.array(img_pil.getchannel(0))
        
        mig = ipt.MIG(img)
        shannon = ipt.ShannonEnt(img)
        selected_mig.append(mig)
        selected_shannon.append(shannon)
    
    # Plot MIG
    plt.figure(figsize=(5, 3))
    plt.plot(test_values, test_mig, 'b-', label=f'Test range [{test_min}, {test_max}]')
    plt.plot(selected_values, selected_mig, 'ro-', label=f'Selected range [{selected_min}, {selected_max}]')
    plt.xlabel(param_name)
    plt.ylabel('MIG')
    plt.legend()
    plt.grid(True)
    plt.show()
    
    # Plot Shannon
    plt.figure(figsize=(5, 3))
    plt.plot(test_values, test_shannon, 'b-', label=f'Test range [{test_min}, {test_max}]')
    plt.plot(selected_values, selected_shannon, 'ro-', label=f'Selected range [{selected_min}, {selected_max}]')
    plt.xlabel(param_name)
    plt.ylabel('Shannon entropy')
    plt.legend()
    plt.grid(True)
    plt.show()

# Study Turing parameters
study_turing_parameter('radius', 1, 50, 1, 40)
study_turing_parameter('rep', 1, 250, 1, 200)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import image_process_tool_box as ipt

plt.style.use(['science', 'no-latex','ieee', 'grid'])

plt.rcParams.update({
    'font.family': 'Calibri',
    'font.size': 16,
    'axes.labelsize': 14,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 10,
    'lines.linewidth': 1.75,
    'lines.markersize': 3,

    # Border styling
    'axes.linewidth': 1.1,     
    'xtick.major.width': 1.1,    
    'ytick.major.width': 1.1,    
    'xtick.minor.width': 0.8,    
    'ytick.minor.width': 0.8,

    'xtick.major.size': 5,      
    'ytick.major.size': 5,
    'xtick.minor.size': 3,      
    'ytick.minor.size': 3,
})

def study_parameter(param_name, test_min, test_max, selected_min, selected_max, is_log_scale=False):
   
   """Study any Perlin noise parameter with test range and selected range by generating images 
   without showing them or saving them"""
   
   # Default config
   defaults = {'scale': 0.05, 
               'octaves': 5, 
               'persistence': 0.5, 
               'lacunarity': 2.0, 
               'texture_function': 'none'}
   
   # Generate test range values
   if is_log_scale:
        test_values = np.exp(np.linspace(np.log(test_min), np.log(test_max), 50))
        selected_values = np.exp(np.linspace(np.log(selected_min), np.log(selected_max), 20))

   else:
        test_values = np.linspace(test_min, test_max, 50)
        selected_values = np.linspace(selected_min, selected_max, 20)
   
   # Calculate MIG and Shannon for exploration range
   test_mig = []
   test_shannon = []

   # Build MIG curve based on changing parameter
   for val in test_values:
       
       # Copy default values to dictionary
       params = defaults.copy()

       # Manipulate specific value in dictionary. 
       params[param_name] = val
       img = ipt.generate_single_perlin_image(250, 250, **params)
       mig = ipt.MIG(img)
       shannon = ipt.ShannonEnt(img)
       test_mig.append(mig)
       test_shannon.append(shannon)
   
   # Calculate MIG and Shannon for working range
   selected_mig = []
   selected_shannon = []

   # Build shannon curve based on parameter
   for val in selected_values:
       
       params = defaults.copy()
       params[param_name] = val
       img = ipt.generate_single_perlin_image(250, 250, **params)
       mig = ipt.MIG(img)
       shannon = ipt.ShannonEnt(img)
       selected_mig.append(mig)
       selected_shannon.append(shannon)
   
   # Plot MIG
   plt.figure(figsize=(5, 3))
   if is_log_scale:
       plt.semilogx(test_values[:-2], test_mig[:-2], 'b-', label=f'Test range [{test_min}, {test_max}]')
       plt.semilogx(selected_values, selected_mig, 'ro-', label=f'Selected range [{selected_min}, {selected_max}]')
   else:
       plt.plot(test_values[:-2], test_mig[:-2], 'b-', label=f'Test range [{test_min}, {test_max}]')
       plt.plot(selected_values, selected_mig, 'ro-', label=f'Selected range [{selected_min}, {selected_max}]')
   
   plt.xlabel(param_name)
   plt.ylabel('MIG')
   plt.legend()
   plt.grid(True)
   plt.show()
   
   # Plot Shannon
   plt.figure(figsize=(5, 3))
   if is_log_scale:
       plt.semilogx(test_values[:-7], test_shannon[:-7], 'b-', label=f'Test range [{test_min}, {test_max}]')
       plt.semilogx(selected_values, selected_shannon, 'ro-', label=f'Selected range [{selected_min}, {selected_max}]')
       plt.grid(True, which="both", ls="--")
   else:
       plt.plot(test_values[:-5], test_shannon[:-5], 'b-', label=f'Test range [{test_min}, {test_max}]')
       plt.plot(selected_values, selected_shannon, 'ro-', label=f'Selected range [{selected_min}, {selected_max}]')
       plt.grid(True)

   plt.xlabel(param_name)
   plt.ylim([6.75,7.5])
   plt.ylabel('Shannon entropy')
   plt.legend()
   plt.show()

# Study all parameters
study_parameter('scale', 0.001, 75, 0.001, 0.45, is_log_scale=True)
study_parameter('octaves', 1, 15, 1, 6)
study_parameter('persistence', 0.01, 10, 0.01, 8)
study_parameter('lacunarity', 0, 18, 0, 10.0)

In [ ]:
# Speckles ()
# The read and blue lines are not aligned exactly because the generator is not perfectly deterministic.
# There are some randomness parameters.

import os
import warnings

os.environ["TQDM_DISABLE"] = "1"
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import image_process_tool_box as ipt
from itertools import product
from speckle_pattern import generate_and_save
import cv2
import matplotlib.pyplot as plt


plt.style.use(['science', 'no-latex','ieee', 'grid'])

plt.rcParams.update({
    'font.family': 'Calibri',
    'font.size': 12,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 8,
    'lines.linewidth': 1.25,
    'lines.markersize': 3,

    # Border styling
    'axes.linewidth': 1.0,     
    'xtick.major.width': 1.0,    
    'ytick.major.width': 1.0,    
    'xtick.minor.width': 0.6,    
    'ytick.minor.width': 0.6,

    'xtick.major.size': 5,      
    'ytick.major.size': 5,
    'xtick.minor.size': 3,      
    'ytick.minor.size': 3,
})


# Speckle pattern parameters
dpi = 24.5

# parameter_ranges = {
#   "speckle_size": (2, 100),                    # Bound study
#   "speckle_density": (dpi, dpi),              # constant
#   "Position": (1.1, 5.0),                     # Natural bound [0,1]
#   "Grid_step": (0.75,1.75),                   # Bound study
#   "size_randomness": (0.1,7.0)                # Natural bound [0,1]
# }

# parameter_ranges = {
#   "speckle_size": (2, 100),                    # Bound study
#   "speckle_density": (dpi, dpi),              # constant
#   "Position": (1.1, 5.0),                     # Natural bound [0,1]
#   "Grid_step": (0.75,1.75),                   # Bound study
#   "size_randomness": (0.1,7.0)                # Natural bound [0,1]
# }




def study_Tspeck_parameter(param_name, test_min, test_max, selected_min, selected_max, is_log_scale=False):
   
   """Study T-speckle parameter with test range and selected range by generating images 
   without showing them or saving them"""
   
   # Default config
   defaults = {
        "dpi": 25.4,
      "speckle_diameter": 9,                   
      "path": None,
      "size_randomness": 0,                # Natural bound [0,1]
      "position_randomness": 0,                     # Natural bound [0,1]   
      "speckle_blur": 0,   
      "grid_step": 2,                 
      
    }
   # Generate test range values
   if is_log_scale:
        test_values = np.exp(np.linspace(np.log(test_min), np.log(test_max), 50))
        selected_values = np.exp(np.linspace(np.log(selected_min), np.log(selected_max), 20))

   else:
        test_values = np.linspace(test_min, test_max, 50)
        selected_values = np.linspace(selected_min, selected_max, 20)
   
   # Calculate MIG and Shannon for exploration range
   test_mig = []
   test_shannon = []

   # Build MIG curve based on changing parameter
   for val in test_values:
       
       # Copy default values to dictionary
       params = defaults.copy()

       # Manipulate specific value in dictionary. 
       params[param_name] = val
       im = generate_and_save(250,250,**params)
       imm_8bit = np.uint8((im/np.max(im))*255)

       print("image type=", type(imm_8bit))
       mig = ipt.MIG(imm_8bit)
       shannon = ipt.ShannonEnt(imm_8bit)
       test_mig.append(mig)
       test_shannon.append(shannon)
   
   # Calculate MIG and Shannon for working range
   selected_mig = []
   selected_shannon = []

   # Build shannon curve based on parameter
   for val in selected_values:
       
       params = defaults.copy()
       params[param_name] = val
       im = generate_and_save(250,250,**params)
       imm_8bit = np.uint8((im/np.max(im))*255)
       print("image type=", type(imm_8bit))
       mig = ipt.MIG(imm_8bit)
       shannon = ipt.ShannonEnt(imm_8bit)
       selected_mig.append(mig)
       selected_shannon.append(shannon)
   
   # Plot MIG
   plt.figure(figsize=(5, 3))
   if is_log_scale:
       plt.semilogx(test_values[:-2], test_mig[:-2], 'b-', label=f'Test range [{test_min}, {test_max}]')
       plt.semilogx(selected_values, selected_mig, 'ro-', label=f'Selected range [{selected_min}, {selected_max}]')
   else:
       plt.plot(test_values[:-2], test_mig[:-2], 'b-', label=f'Test range [{test_min}, {test_max}]')
       plt.plot(selected_values, selected_mig, 'ro-', label=f'Selected range [{selected_min}, {selected_max}]')
   
   plt.xlabel(param_name)
   plt.ylabel('MIG')
   plt.legend()
   plt.grid(True)
   plt.show()
   
   # Plot Shannon
   plt.figure(figsize=(5, 3))
   if is_log_scale:
       plt.semilogx(test_values[:-7], test_shannon[:-7], 'b-', label=f'Test range [{test_min}, {test_max}]')
       plt.semilogx(selected_values, selected_shannon, 'ro-', label=f'Selected range [{selected_min}, {selected_max}]')
       plt.grid(True, which="both", ls="--")
   else:
       plt.plot(test_values[:-5], test_shannon[:-5], 'b-', label=f'Test range [{test_min}, {test_max}]')
       plt.plot(selected_values, selected_shannon, 'ro-', label=f'Selected range [{selected_min}, {selected_max}]')
       plt.grid(True)

   plt.xlabel(param_name)
   plt.ylim([0,0.75])
   plt.ylabel('Shannon entropy')
   plt.legend()
   plt.show()

study_Tspeck_parameter('grid_step', 1, 50, 1, 25)

In [ ]:
# Position randomness
# The read and blue lines are not aligned exactly because the generator is not perfectly deterministic.
# There are some randomness parameters.

import os
import warnings

os.environ["TQDM_DISABLE"] = "1"
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import image_process_tool_box as ipt
from itertools import product
from speckle_pattern import generate_and_save
import cv2



# Speckle pattern parameters
dpi = 24.5

parameter_ranges = {
  "speckle_size": (2, 100),                    # Bound study
  "speckle_density": (dpi, dpi),              # constant
  "Position": (1.1, 5.0),                     # Natural bound [0,1]
  "Grid_step": (0.75,1.75),                   # Bound study
  "size_randomness": (0.1,7.0)                # Natural bound [0,1]
}

height = 500
width = 2000

# Full exploration range [0, 1]
test_param_full = np.linspace(0, 20)
mig_values_full = []
shannon_values_full = []

# Selected range [2, 30]
test_param_selected = np.linspace(0, 10)
mig_values_selected = []
shannon_values_selected = []

for i in range(len(test_param_full)):
  img = generate_and_save(
      height, 
      width, 
      dpi=dpi, 
      speckle_diameter = 15,
      path = 'Boundary_test.tiff', 
      size_randomness= 0,
      position_randomness= test_param_full[i], 
      speckle_blur=1.5, 
      grid_step = 1
      ) 

  imasg = cv2.imread('Boundary_test.tiff', cv2.IMREAD_GRAYSCALE)
  mig = ipt.MIG(imasg)
  shannon = ipt.ShannonEnt(imasg)
  mig_values_full.append(mig)
  shannon_values_full.append(shannon)

# Selected range
for i in range(len(test_param_selected)):
  img = generate_and_save(
      height, 
      width, 
      dpi=dpi, 
      speckle_diameter = 15,
      path = 'Boundary_test.tiff', 
      size_randomness= 0,
      position_randomness= test_param_selected[i], 
      speckle_blur=1.5, 
      grid_step = 1
      ) 

  imasg = cv2.imread('Boundary_test.tiff', cv2.IMREAD_GRAYSCALE)
  mig = ipt.MIG(imasg)
  shannon = ipt.ShannonEnt(imasg)
  mig_values_selected.append(mig)
  shannon_values_selected.append(shannon)

# Plot MIG values
plt.figure()
plt.plot(test_param_full, mig_values_full, 'b-', label='Full range [0, 20]')
plt.plot(test_param_selected, mig_values_selected, 'ro-', label='Selected range [0, 10]')
plt.xlabel("Position randomness")
plt.ylabel("MIG")
plt.legend()
plt.grid(True)
plt.show()

# Plot Shannon entropy values
plt.figure()
plt.plot(test_param_full, shannon_values_full, 'b-', label='Full range [0, 20]')
plt.plot(test_param_selected, shannon_values_selected, 'ro-', label='Selected range [0, 10]')
plt.xlabel("Position randomness")
plt.ylabel("Shannon entropy")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Grid step
# The read and blue lines are not aligned exactly because the generator is not perfectly deterministic.
# There are some randomness parameters.

import os
import warnings

os.environ["TQDM_DISABLE"] = "1"
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import image_process_tool_box as ipt
from itertools import product
from speckle_pattern import generate_and_save
import cv2



# Speckle pattern parameters
dpi = 24.5

parameter_ranges = {
  "speckle_size": (2, 100),                    # Bound study
  "speckle_density": (dpi, dpi),              # constant
  "Position": (1.1, 5.0),                     # Natural bound [0,1]
  "Grid_step": (0.75,1.75),                   # Bound study
  "size_randomness": (0.1,7.0)                # Natural bound [0,1]
}

height = 500
width = 2000

# Full exploration range [0, 1]
test_param_full = np.linspace(0.1, 25)
mig_values_full = []
shannon_values_full = []

# Selected range [2, 30]
test_param_selected = np.linspace(0.5, 13)
mig_values_selected = []
shannon_values_selected = []

for i in range(len(test_param_full)):
  img = generate_and_save(
      height, 
      width, 
      dpi=dpi, 
      speckle_diameter = 7,
      path = 'Boundary_test.tiff', 
      size_randomness= 0,
      position_randomness= 0, 
      speckle_blur=1.5, 
      grid_step = test_param_full[i]
      ) 

  imasg = cv2.imread('Boundary_test.tiff', cv2.IMREAD_GRAYSCALE)
  mig = ipt.MIG(imasg)
  shannon = ipt.ShannonEnt(imasg)
  mig_values_full.append(mig)
  shannon_values_full.append(shannon)

# Selected range
for i in range(len(test_param_selected)):
  img = generate_and_save(
      height, 
      width, 
      dpi=dpi, 
      speckle_diameter = 7,
      path = 'Boundary_test.tiff', 
      size_randomness= 0,
      position_randomness= 0, 
      speckle_blur=1.5, 
      grid_step = test_param_selected[i]
      ) 

  imasg = cv2.imread('Boundary_test.tiff', cv2.IMREAD_GRAYSCALE)
  mig = ipt.MIG(imasg)
  shannon = ipt.ShannonEnt(imasg)
  mig_values_selected.append(mig)
  shannon_values_selected.append(shannon)

# Plot MIG values
plt.figure()
plt.plot(test_param_full, mig_values_full, 'b-', label='Full range [0, 25]')
plt.plot(test_param_selected, mig_values_selected, 'ro-', label='Selected range [0, 13]')
plt.xlabel("Grid step")
plt.ylabel("MIG")
plt.legend()
plt.grid(True)
plt.show()

# Plot Shannon entropy values
plt.figure()
plt.plot(test_param_full, shannon_values_full, 'b-', label='Full range [0, 25]')
plt.plot(test_param_selected, shannon_values_selected, 'ro-', label='Selected range [0, 13]')
plt.xlabel("Grid step")
plt.ylabel("Shannon entropy")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import os
import warnings

os.environ["TQDM_DISABLE"] = "1"
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import image_process_tool_box as imgan
from itertools import product
from speckle_pattern import generate_and_save
import cv2

# Turing transformation parameter test
height = 500
width = 2000

# Generate and save a base pattern
base_pattern_path = 'base_pattern_test.tif'
base_pattern = imgan.generate_single_perlin_image(
    height, width, 
    scale=50, octaves=6, persistence=0.5, lacunarity=2.0, 
    texture_function='sinusoidal'
)
cv2.imwrite(base_pattern_path, base_pattern)

# Test radius (reps = 25 constant)
test_radius = np.linspace(1, 15, 20)
mig_radius = []
shannon_radius = []

for radius in test_radius:
    img_pil = imgan.make_turing_single(base_pattern_path, rep=25, radius=int(radius))
    img = np.array(img_pil.getchannel(0))
    
    mig_radius.append(imgan.MIG(img))
    shannon_radius.append(imgan.ShannonEnt(img))

# Test repetitions (radius = 5 constant)
test_reps = np.linspace(5, 50, 20)
mig_reps = []
shannon_reps = []

for rep in test_reps:
    img_pil = imgan.make_turing_single(base_pattern_path, rep=int(rep), radius=5)
    img = np.array(img_pil.getchannel(0))
    
    mig_reps.append(imgan.MIG(img))
    shannon_reps.append(imgan.ShannonEnt(img))

# Plot
plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
plt.plot(test_radius, mig_radius, 'b-o')
plt.xlabel("Radius")
plt.ylabel("MIG")
plt.grid(True)

plt.subplot(2, 2, 2)
plt.plot(test_radius, shannon_radius, 'r-o')
plt.xlabel("Radius")
plt.ylabel("Shannon Entropy")
plt.grid(True)

plt.subplot(2, 2, 3)
plt.plot(test_reps, mig_reps, 'b-o')
plt.xlabel("Repetitions")
plt.ylabel("MIG")
plt.grid(True)

plt.subplot(2, 2, 4)
plt.plot(test_reps, shannon_reps, 'r-o')
plt.xlabel("Repetitions")
plt.ylabel("Shannon Entropy")
plt.grid(True)

plt.tight_layout()
plt.show()

os.remove(base_pattern_path)

# General image processing

In [9]:
import os
import matplotlib.pyplot as plt
import cv2
import image_process_tool_box as iat
import numpy as np
import random as rand
from speckle_pattern import generate_and_save
from scipy import integrate
import math
import torch
# from torch_fourier_shift.fourier_shift_dft import fourier_shift_dft_2d
import matplotlib.pyplot as plt

plt.style.use(['science', 'no-latex','ieee', 'grid'])

plt.rcParams.update({
    'font.family': 'Calibri',
    'font.size': 16,
    'axes.labelsize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 7,
    'lines.linewidth': 1.25,
    'lines.markersize': 3,

    # Border styling
    'axes.linewidth': 1.0,     
    'xtick.major.width': 1.0,    
    'ytick.major.width': 1.0,    
    'xtick.minor.width': 0.6,    
    'ytick.minor.width': 0.6,

    'xtick.major.size': 5,      
    'ytick.major.size': 5,
    'xtick.minor.size': 3,      
    'ytick.minor.size': 3,
})

numbers_of_interest = [6,10,14]

for batch in [0,2,3,4,5,6,8,9,10,11]:

    # For each batch, get subsets corresponding to the image of interest numbers
    for pattern_number in numbers_of_interest:

        print(f"\nBatch {batch}\n--------------")
        image_folder = rf"data\speckle_pattern_img\reference_im\ref_{batch}"

        
        im_typ = 'tif'
        image_files = iat.get_image_strings(image_folder, 
                                            imagetype=im_typ,  
                                            parity='even')
        print((image_files))

        hist_frequen_save_folder = rf"output\plots\article_subsets\image_{pattern_number}"
        frequen_save_folder = rf"output\plots\article_freq2\image_{pattern_number}\power_spectrums"

        if not os.path.exists(hist_frequen_save_folder):
            os.makedirs(hist_frequen_save_folder)
        if not os.path.exists(frequen_save_folder):
            os.makedirs(frequen_save_folder)

        # String management for saving plots and figures
        myfile = image_files[int(pattern_number/2)]
        parts = myfile.split('_')
        hitogram_name = '_'.join([parts[0],parts[1],f"{batch}","hist.png"])
        power_name = '_'.join([parts[0],parts[1],f"{batch}","power.png"])
        print(f"\nString parts: {myfile}\n")

        # Image
        image_path = os.path.join(image_folder, myfile)
        # print('\nspecific path:', image_path)
        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        # image = cv2.imread(r"C:\Users\General User\nokop\pattern1\data\speckle_pattern_img\reference_im\6_Generated_spec_image.tif",cv2.IMREAD_GRAYSCALE)
        print(f"Image shape {image.shape}")

        # Histogram: create and save to folder
        print(image.shape)
        plt.figure(figsize=(4,3))
        plt.hist(image.ravel(), 256, [0, 256], color="black") 
        plt.ylabel("Frequency")
        plt.ylim(0,50000)
        plt.xlabel("Grey level intensity")
        histy_savey = os.path.join(hist_frequen_save_folder,hitogram_name)
        plt.savefig(histy_savey)
        # plt.show()

        # Spectral analysis
        power_of_the_darkside = iat.uncentered_spec_power(image,
                                                        plot_type = "semilog",
                                                        plot = False)
        luke_I_am_your_father = os.path.join(frequen_save_folder,power_name)
        plt.savefig(luke_I_am_your_father)
        
        # plt.show()

        # Save image subset in the same folder
        subsets = iat.img_subsets(image, 250,25)
        subset = subsets[0,0,:,:]
        plt.imshow( subset, cmap = 'grey')
        plt.grid(False)
        plt.ylabel(f"$y$ [pixel]")
        plt.xlabel(f"$x$ Pixels")
        plt.grid(False)
        plt.axis("on")
        to_sav = os.path.join(hist_frequen_save_folder,f'Image_{pattern_number}_b{batch}.png')
        plt.imsave(to_sav,subset,cmap='grey')
        # plt.show(block=False)
        plt.close()

        
        plt.style.use(['science', 'no-latex','ieee', 'grid'])

        plt.rcParams.update({
            'font.family': 'Calibri',
            'font.size': 16,
            'axes.labelsize': 14,
            'xtick.labelsize': 10,
            'ytick.labelsize': 10,
            'legend.fontsize': 7,
            'lines.linewidth': 1.25,
            'lines.markersize': 3,

            # Border styling
            'axes.linewidth': 1.0,     
            'xtick.major.width': 1.0,    
            'ytick.major.width': 1.0,    
            'xtick.minor.width': 0.6,    
            'ytick.minor.width': 0.6,

            'xtick.major.size': 5,      
            'ytick.major.size': 5,
            'xtick.minor.size': 3,      
            'ytick.minor.size': 3,
        })




Batch 0
--------------
['0_Generated_spec_image_turing.tif', '2_Generated_spec_image_turing.tif', '4_Generated_spec_image_turing.tif', '6_Generated_spec_image_turing.tif', '8_Generated_spec_image_turing.tif', '10_Generated_spec_image_turing.tif', '12_Generated_spec_image_turing.tif', '14_Generated_spec_image_turing.tif', '16_Generated_spec_image_turing.tif', '18_Generated_spec_image_turing.tif']

String parts: 6_Generated_spec_image_turing.tif

Image shape (500, 2000)
(500, 2000)


C:\Users\General User\AppData\Local\Temp\ipykernel_7988\313452428.py:81: MatplotlibDeprecationWarning: Passing the range parameter of hist() positionally is deprecated since Matplotlib 3.9; the parameter will become keyword-only in 3.11.
  plt.hist(image.ravel(), 256, [0, 256], color="black")
C:\Users\General User\AppData\Local\Temp\ipykernel_7988\313452428.py:101: UserWarning: Attempt to set non-positive xlim on a log-scaled axis will be ignored.
  plt.imshow( subset, cmap = 'grey')



Batch 0
--------------
['0_Generated_spec_image_turing.tif', '2_Generated_spec_image_turing.tif', '4_Generated_spec_image_turing.tif', '6_Generated_spec_image_turing.tif', '8_Generated_spec_image_turing.tif', '10_Generated_spec_image_turing.tif', '12_Generated_spec_image_turing.tif', '14_Generated_spec_image_turing.tif', '16_Generated_spec_image_turing.tif', '18_Generated_spec_image_turing.tif']

String parts: 10_Generated_spec_image_turing.tif

Image shape (500, 2000)
(500, 2000)

Batch 0
--------------
['0_Generated_spec_image_turing.tif', '2_Generated_spec_image_turing.tif', '4_Generated_spec_image_turing.tif', '6_Generated_spec_image_turing.tif', '8_Generated_spec_image_turing.tif', '10_Generated_spec_image_turing.tif', '12_Generated_spec_image_turing.tif', '14_Generated_spec_image_turing.tif', '16_Generated_spec_image_turing.tif', '18_Generated_spec_image_turing.tif']

String parts: 14_Generated_spec_image_turing.tif

Image shape (500, 2000)
(500, 2000)

Batch 2
--------------
['

In [ ]:
import image_process_tool_box as ipt
import os
import cv2
import matplotlib.pyplot as plt

# image_folder = rf"C:\Users\General User\nokop\pattern1\data\speckle_pattern_img\reference_im\hold_ref_noised\ref_11"
image_folder = rf"C:\Users\General User\nokop\pattern1\data\speckle_pattern_img\reference_im\newPerlin_hold_folder\sinusoidal"
images_lis = [180,60,36,132,22]
ind = [i // 2 for i in images_lis] 
print(ind)

image_listt = []

for index in ind:
    im_typ = 'tif'
    image_files = ipt.get_image_strings(image_folder, imagetype=im_typ,  parity='even')
    # print((image_files))

    # String management for saving plots
    myfile = image_files[index]
    print(f'My file: {myfile}')
    parts = myfile.split('_')

    image_path = os.path.join(image_folder, myfile)
    # print('\nspecific path:', image_path)
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    image_listt.append(image)

    # print(image.shape)
    # plt.hist(image.ravel(),253,[1,253], color="black")
    # plt.ylabel("Frequency")
    # plt.xlabel("Grey level intensity")
    # plt.show()

# append some bad image
# batch = 6
# img_prefix = 1092
# prefix2index = int(img_prefix / 2)
# print(prefix2index)
# bad_img_path = rf"data\speckle_pattern_img\reference_im\hold_ref_noised\ref_{batch}"
# list_of_random_images = ipt.get_image_strings(bad_img_path)
# randim_file_path = os.path.join(bad_img_path, list_of_random_images[prefix2index])
# random_image = cv2.imread(randim_file_path,cv2.IMREAD_GRAYSCALE)

# Append to list
# image_listt.append(random_image)

legende = ["Best optimised", "Second best","Third best","Fourth best","Fifth best"]

power_of_the_darkside = ipt.spec_power_superimpose(image_listt, plot_type="semilog",plot=False)
plt.ylim(1e6,1e12)
plt.legend(
    legende, 
    loc='upper center', 
    bbox_to_anchor=(0.5, -0.2), 
    ncol=2
    )
plt.show()


In [ ]:


import cv2
import os
import matplotlib.pyplot as plt
import image_process_tool_box as ipt

# plt.style.use(['science', 'no-latex','ieee', 'grid'])

# plt.rcParams.update({
#     'font.family': 'Calibri',
#     'font.size': 16,
#     'axes.labelsize': 16,
#     'xtick.labelsize': 11,
#     'ytick.labelsize': 11,
#     'legend.fontsize': 11,
#     'lines.linewidth': 1.25,
#     'lines.markersize': 3,

#     # Border styling
#     'axes.linewidth': 1.0,     
#     'xtick.major.width': 1.0,    
#     'ytick.major.width': 1.0,    
#     'xtick.minor.width': 0.6,    
#     'ytick.minor.width': 0.6,

#     'xtick.major.size': 5,      
#     'ytick.major.size': 5,
#     'xtick.minor.size': 3,      
#     'ytick.minor.size': 3,
# })

analay = 3

wcb = False

if wcb:
    if analay == 0:
        # Best transformed
        freq_file = "Best_transformed.png"
        image_num = [1084,74,674,58,1094,246]
        batch = [0,1,2,3,4,5]
        legende = ["RS", "RCh","RP","RPc", "RPbl","RPs"]
        # legende = ["Best Class 7","Best Class 9", "Best Class 10","Best Class 11", "Best Class 12"]

    if analay == 1:
        # Worst transformed
        freq_file = "Worst_transformed.png"
        image_num = [694,932,976,488,1024,86]
        batch = [0,1,2,3,4,5]
        legende = ["RS", "RCh","RP","RPc", "RPbl","RPs"]
        # legende = ["Worst Class 7","Worst Class 9", "Worst Class 10","Worst Class 11", "Worst Class 12"]


    if analay == 2:
        # Best untransformed
        freq_file = "Best_untransformed.png"
        image_num = [1092,8,298,306,384,278]
        batch = [6,7,8,9,10,11]
        legende = ["S", "Ch","P","Pc", "Pbl","Ps"]
        # legende = ["Best Class 1","Best Class 3", "Best Class 4","Best Class 5", "Best Class 6"]


    if analay == 3:
        # Worst untransformed
        freq_file = "Worst_untransformed.png"
        image_num = [124,862,626,946,340,28]
        batch = [6,7,8,9,10,11]
        legende = ["S","Ch", "P","Pc", "Pbl","Ps"]
        # legende = ["Worst Class 1", "Worst Class 2","Worst Class 3", "Worst Class 4","Worst Class 5", "Best Class 6"]

else:
    if analay == 0:
        # Best transformed
        freq_file = "Best_transformed.png"
        image_num = [1084,674,58,1094,246]
        batch = [0,2,3,4,5]
        legende = ["RS","RP","RPc", "RPbl","RPs"]
        # legende = ["Best Class 7","Best Class 9", "Best Class 10","Best Class 11", "Best Class 12"]

    if analay == 1:
        # Worst transformed
        freq_file = "Worst_transformed.png"
        image_num = [694,976,488,1024,86]
        batch = [0,2,3,4,5]
        legende = ["RS","RP","RPc", "RPbl","RPs"]
        # legende = ["Worst Class 7","Worst Class 9", "Worst Class 10","Worst Class 11", "Worst Class 12"]


    if analay == 2:
        # Best untransformed
        freq_file = "Best_untransformed.png"
        image_num = [1092,298,306,384,278]
        batch = [6,8,9,10,11]
        legende = ["S","P","Pc", "Pbl","Ps"]
        # legende = ["Best Class 1","Best Class 3", "Best Class 4","Best Class 5", "Best Class 6"]


    if analay == 3:
        # Worst untransformed
        freq_file = "Worst_untransformed.png"
        image_num = [124,626,946,340,28]
        batch = [6,8,9,10,11]
        legende = ["S", "P","Pc", "Pbl","Ps"]
        # legende = ["Worst Class 1", "Worst Class 2","Worst Class 3", "Worst Class 4","Worst Class 5", "Best Class 6"]





ppp = rf"C:\Users\General User\nokop\pattern2\data\speckle_pattern_img\reference_im\hold_ref_noised"

image_list = []

for image_numm, bat in zip(image_num, batch):
    immm = ipt.get_batch_image(ppp, bat, image_numm)
    image_list.append(immm)


frequency_pathways = r"output\plots\power_spectra"
if not os.path.exists(frequency_pathways):
    os.makedirs(frequency_pathways)
save_frequencies = os.path.join(frequency_pathways,freq_file)

fiiiiig = ipt.spec_power_superimpose(image_list, plot_type="semilog", plot=False)
plt.ylim(1e6,1e14)
plt.legend(
    legende, 
    loc='upper center', 
    bbox_to_anchor=(0.5, -0.2), 
    ncol=3
    )
plt.savefig(save_frequencies)
plt.show()

In [ ]:
import image_process_tool_box as ipt
import cv2
import os
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from scipy.signal import find_peaks


batch1 = 6
batch2 = 11

image_number1 = 1114
image_folder1 = rf"data\speckle_pattern_img\reference_im\hold_ref_noised\ref_{batch1}"
image_files1 = ipt.get_image_strings(image_folder1, imagetype='tif',  parity='even')
path_to_image1 = os.path.join(image_folder1,image_files1[int(image_number1/2)])
# print(f"Loading image {image_files1[int(image_number1/2)]}")
image_1 = cv2.imread(path_to_image1, cv2.IMREAD_GRAYSCALE)



image_number2 = 286
image_folder2 = rf"data\speckle_pattern_img\reference_im\hold_ref_noised\ref_{batch2}"
image_files2 = ipt.get_image_strings(image_folder2, imagetype='tif',  parity='even')
path_to_image2 = os.path.join(image_folder2,image_files2[int(image_number2/2)])
image_2 = cv2.imread(path_to_image2, cv2.IMREAD_GRAYSCALE)

ppp = r"data\speckle_pattern_img\reference_im\Validation\218_Generated_spec_image.tif"
# image_2 = cv2.imread(ppp, cv2.IMREAD_GRAYSCALE)

plt.imshow(image_1,cmap='grey')
plt.grid('off')
plt.axis(False)
plt.show()

print(image_1.shape)
greys = image_2[image_1.shape[0]//2, :]  
xs = np.arange(0, image_1.shape[1])  # Horizontal positions
print(greys.shape)

plt.figure(figsize=(6,3))
plt.plot(xs, greys)
plt.xlabel('Horizontal position [pixel]')
plt.ylabel('Grey level intensity')
plt.ylim([0,265])
plt.show()



print("\n--------------------\nDecentered spectrum\n------------")
fiiiiig = ipt.spec_power_superimpose([image_1,image_2],  plot_type = "semilog",plot=False)
plt.legend(
    ["Figure 2.1(a)","Figure 2.1(b)"], 
    loc='upper center', 
    bbox_to_anchor=(0.5, -0.2), 
    ncol=2
    )
# plt.ylim(0,1e9)
plt.show()


In [28]:
import image_process_tool_box as ipt
import numpy as np

# Read optimised excel sheet
foplder = 1
error_column = 0
batch = 0
print(f'------------\n{batch}')

if foplder == 1:
    excel_path = rf"output\excel_docs\excel_{batch}"
else:
    # excel_path = r'output\excel_docs'
    excel_path = r"output\excel_docs\after_corr_sinusoidal"

op_metrics, omeas_error, op_param, onans, oindicators = ipt.read_spec_excel(
    excel_path, 
    # doc_num = 11,
    # doc_name='corrected_batch'
    )

# Get top 5 RMSE, bias and std. dev error
mask = (omeas_error[:,0] != 0) 
error = np.abs(omeas_error[mask,error_column])
error_analys = True


analysisisi = 12

if analysisisi == 1:
    get_highest = True 
else:
    get_highest = False
amm = 2

if error_analys:
    # get indices of 5 smallest or largest values
    if get_highest:
        highest_idx = np.argsort(error)[-amm:]
        selected_idx = highest_idx
        label = "Highest"
    else:
        lowest_idx = np.argsort(error)[:amm]
        selected_idx = lowest_idx
        label = "Lowest"

    selected_values = error[selected_idx]
    original_indices = np.where(mask)[0][selected_idx]

    print(f"\n{label} errors:", selected_values)
    print("\n--------------\nTheir image numbers:", original_indices * 2)
else:
    metric_column = 7
    metric = np.abs(op_metrics[mask, metric_column])

    if get_highest:
        highest_idx = np.argsort(metric)[-amm:]
        selected_idx = highest_idx
        label = "Highest"
    else:
        lowest_idx = np.argsort(metric)[:amm]
        selected_idx = lowest_idx
        label = "Lowest"

    selected_values = metric[selected_idx]
    original_indices = np.where(mask)[0][selected_idx]

    print(f"\n{label} metric values:", selected_values)
    print("\n--------------\nTheir image numbers:", original_indices * 2)


# Function
def glean_extreme_values(excel_path, batch_type = True, column = 0,  error_analys = True, 
                         get_highest = False, batch=11, count = 5):

    """ 
    Analyse excel column and get highest or lowest entries including their indexes
    """
    column = 0
    print(f'------------\n{batch}')

    if batch_type:
        excel_path = rf"C:\Users\General User\nokop\pattern2\output\excel_docs\excel_{batch}"
   

    op_metrics, omeas_error, _, _, _ = ipt.read_spec_excel(excel_path, doc_num = None)

    # Get top 5 RMSE, bias and std. dev error
    mask = (omeas_error[:,0] != 0) 
    error = np.abs(omeas_error[mask,error_column])

    if error_analys:
        # get indices of 5 smallest or largest values
        if get_highest:
            highest_idx = np.argsort(error)[-count:]
            selected_idx = highest_idx
            label = "Highest"
        else:
            lowest_idx = np.argsort(error)[:count]
            selected_idx = lowest_idx
            label = "Lowest"

        selected_values = error[selected_idx]
        original_indices = np.where(mask)[0][selected_idx]

        print(f"\n{label} errors:", selected_values)
        print("\n--------------\nTheir image numbers:", original_indices * 2)
    else:
        column = 5
        metric = np.abs(op_metrics[mask, metric_column])

        if get_highest:
            highest_idx = np.argsort(metric)[-count:]
            selected_idx = highest_idx
            label = "Highest"
        else:
            lowest_idx = np.argsort(metric)[:count]
            selected_idx = lowest_idx
            label = "Lowest"

        selected_values = metric[selected_idx]
        original_indices = np.where(mask)[0][selected_idx]

        print(f"\n{label} metric values:", selected_values)
        print("\n--------------\nTheir image numbers:", original_indices * 2)



------------
0
Current doc_number (Error analysis): 12

Lowest errors: [0.00090901 0.0009264 ]

--------------
Their image numbers: [1084  520]


In [ ]:


import cv2
import os
import matplotlib.pyplot as plt
import image_process_tool_box as ipt


plt.style.use(['science', 'no-latex', 'ieee', 'grid'])

# Update font and line settings
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 12,
    'axes.labelsize': 8,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 7,
    'lines.linewidth': 1,
    'lines.markersize': 3
})


analay = 2

if analay == 0:
    # Best transformed
    freq_file = "Best_tcb.png"
    image_num = [74]
    batch = [1]
    # legende = ["S", "Ch","P","Pc", "Pbl","Ps"]

if analay == 1:
    # Worst transformed
    freq_file = "Worst_tcb.png"
    image_num = [168]
    batch = [1]
    # legende = ["S","Ch", "P","Pc", "Pbl","Ps"]

if analay == 2:
    # Best untransformed
    freq_file = "Best_utcb.png"
    image_num = [278,602,24]
    batch = [11,11,11]
    # legende = ["S","Ch", "P","Pc", "Pbl","Ps"]

if analay == 3:
    # Worst untransformed
    freq_file = "Worst_utcb.png"
    image_num = [1194]
    batch = [7]
    # legende = ["S","Ch", "P","Pc", "Pbl","Ps"]

#--------------------------------------------------------------
# if analay == 0:
#     # Best transformed
#     freq_file = "Best_transformedcb.png"
#     image_num = [1084,74,674,58,1094,246]
#     batch = [0,1,2,3,4,5]
#     legende = ["S", "Ch","P","Pc", "Pbl","Ps"]

# if analay == 1:
#     # Worst transformed
#     freq_file = "Worst_transformedcb.png"
#     image_num = [782,168,1138,812,430,1092]
#     batch = [0,1,2,3,4,5]
#     legende = ["S","Ch", "P","Pc", "Pbl","Ps"]

# if analay == 2:
#     # Best untransformed
#     freq_file = "Best_untransformedcb.png"
#     image_num = [1092,8,298,306,384,602]
#     batch = [6,7,8,9,10,11]
#     legende = ["S","Ch", "P","Pc", "Pbl","Ps"]

# if analay == 3:
#     # Worst untransformed
#     freq_file = "Worst_untransformedcb.png"
#     image_num = [1094,1194,960,286,1078,764]
#     batch = [6,7,8,9,10,11]
#     legende = ["S","Ch", "P","Pc", "Pbl","Ps"]

#------------------------------------------------------------
# if analay == 0:
#     # Best transformed
#     freq_file = "Best_transformed.png"
#     image_num = [1084,674,58,1094,246]
#     batch = [0,2,3,4,5]
#     legende = ["S", "P","Pc", "Pbl","Ps"]
# if analay == 1:
#     # Worst transformed
#     freq_file = "Worst_transformed.png"
#     image_num = [782,1138,812,430,1092]
#     batch = [0,2,3,4,5]
#     legende = ["S", "P","Pc", "Pbl","Ps"]

# if analay == 2:
#     # Best untransformed
#     freq_file = "Best_untransformed.png"
#     image_num = [1092,298,306,384,602]
#     batch = [6,8,9,10,11]
#     legende = ["S", "P","Pc", "Pbl","Ps"]

# if analay == 3:
#     # Worst untransformed
#     freq_file = "Worst_untransformed.png"
#     image_num = [1094,960,286,1078,764]
#     batch = [6,8,9,10,11]
#     legende = ["S", "P","Pc", "Pbl","Ps"]



# if analay == 2:
#     # Best untransformed
#     freq_file = "Best_untransformed.png"
#     image_num = [602,752,276]
#     batch = [11,11,11]
#     legende = ["Subset size 17", "Subset size 33", "Subset size 67"]



# if analay == 3:
#     # Worst untransformed
#     freq_file = "Worst_untransformed.png"
#     image_num = [26000,286]
#     batch = [12,10]
#     legende = ["Image (a)", "Image (b)"]




ppp = rf"C:\Users\General User\nokop\pattern2\data\speckle_pattern_img\reference_im\hold_ref_noised"

image_list = []

for image_numm, bat in zip(image_num, batch):
    immm = ipt.get_batch_image(ppp, bat, image_numm)
    image_list.append(immm)


# for i, img in enumerate(image_list):
#     print(img.shape)
#     plt.figure()
#     plt.imshow(img, cmap='gray')
#     plt.axis('on')
    
#     plt.figure(figsize=(4,2))
#     plt.plot(img[img.shape[0]//2, :], color='black')
#     # plt.title(f'Slice at y={img.shape[0]//2}')
#     plt.ylabel("Grey level intensity")
#     plt.xlabel("Horzontal position [pixel]")
#     plt.ylim([0,265])
#     plt.show()

#     ipt.uncentered_spec_power(img)
#     plt.ylim(1e-6,10)
#     plt.show()
# plt.show()

frequency_pathways = r"output\plots\frequency_comparison2"
if not os.path.exists(frequency_pathways):
    os.makedirs(frequency_pathways)

save_frequencies = os.path.join(frequency_pathways,freq_file)


fiiiiig = ipt.spec_power_superimpose(image_list, plot_type="semilog", plot=False)
plt.ylim(1e5,1e14)
# plt.legend(
#     # legende, 
#     loc='upper center', 
#     bbox_to_anchor=(0.5, -0.2), 
#     ncol=3
#     )
plt.savefig(save_frequencies)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def f(x):
    return 3 * np.exp(-x) * np.sin(x)

# Sample points
x_samples = np.array([0, 1, 2, 3, 4, 5])
y_samples = f(x_samples)

# Create step points for NN interpolation
x_steps = []
y_steps = []
for i in range(len(x_samples)):
    # Add point before jump
    x_steps.append(x_samples[i] - 0.5 if i > 0 else x_samples[i])
    y_steps.append(y_samples[i])
    # Add point after jump
    x_steps.append(x_samples[i] + 0.5 if i < len(x_samples)-1 else x_samples[i])
    y_steps.append(y_samples[i])

# True function
x_fine = np.linspace(0, 5, 500)
y_true = f(x_fine)

# Plotting
plt.figure(figsize=(10, 6))
plt.plot(x_fine, y_true, 'b-', alpha=0.5, label='True function')
plt.plot(x_samples, y_samples, 'ko', markersize=8, label='Samples')
plt.plot(x_steps, y_steps, 'r-', drawstyle='steps-pre', linewidth=2, label='Nearest-neighbor')

plt.xlabel('x')
plt.ylabel('y')
plt.title('Nearest-neighbor Interpolation')
plt.legend()
plt.grid(True)
plt.show()

#### Statistical metric analysis

In [ ]:
import image_process_tool_box as ipt
import matplotlib.pyplot as plt
import numpy as np

batch = 5
path2excel = rf"output\excel_docs\excel_{batch}"

p_metrics, meas_error, p_param, nans, indicators = ipt.read_spec_excel(path2excel, doc_num = None)

d = p_metrics[:,1]
# MIG boxplot
fig = plt.figure(figsize =(5, 3))
ax = fig.add_axes([0, 0, 1, 1])
bp = ax.boxplot(d)
plt.xlabel("MIG boxplot for RTG-Perlin-sinusoidal")
plt.ylabel("MIG")

plt.show()



In [ ]:
# Get RMSE elbow to see where it gets significantly worse 
# after optimisation

import image_process_tool_box as ipt
import matplotlib.pyplot as plt
import numpy as np

plt.style.use(['science', 'no-latex','ieee', 'grid'])

plt.rcParams.update({
    'font.family': 'Calibri',
    'font.size': 12,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 11,
    'lines.linewidth': 1.25,
    'lines.markersize': 3,

    # Border styling
    'axes.linewidth': 1.0,     
    'xtick.major.width': 1.0,    
    'ytick.major.width': 1.0,    
    'xtick.minor.width': 0.6,    
    'ytick.minor.width': 0.6,

    'xtick.major.size': 5,      
    'ytick.major.size': 5,
    'xtick.minor.size': 3,      
    'ytick.minor.size': 3,
})
optimised_patterns_data = r'output\excel_docs'

op_metrics, omeas_error, op_param, onans, oindicators = ipt.read_spec_excel(
    optimised_patterns_data, 
    doc_num = None)

errors = omeas_error[:, 0]
mask = (~np.isnan(errors)) & (errors != 0) & (np.abs(errors - np.nanmean(errors)) <= 3 * np.nanstd(errors))


# RMSEs for specific patterns
ascending_rmse = np.sort(errors[mask])

sorted_indices = np.where(errors[mask]==ascending_rmse)
print(f"\nSorted indices: {ascending_rmse.shape}\n")

domain = np.arange(1, len(ascending_rmse) + 1)  

plt.scatter(1, 0.0004061, marker="*", s=22, label="Best overall pattern")

plt.plot(domain[:10], ascending_rmse[:10], color = 'black',marker='o', markersize=1)
plt.xlabel('Rank')
plt.ylabel('RMSE')
# plt.title('RMSE Elbow Plot - Optimization Results')
plt.grid(True, alpha=0.3)
plt.legend(loc='upper center', 
    bbox_to_anchor=(0.5, -0.2))
plt.tight_layout()
plt.show()

# Optional: Print some statistics to help identify the elbow
print(f"Total configurations after filtering: {len(ascending_rmse)}")
print(f"Best RMSE: {ascending_rmse[0]:.6f}")
print(f"Median RMSE: {np.median(ascending_rmse):.6f}")
print(f"Worst RMSE: {ascending_rmse[-1]:.6f}")

#### Improvement on checkerboard

In [ ]:
# This could be an improvement on the checkerboard pattern
# It retains the contrast and does away with the repetition

import matplotlib.pyplot as plt
from PIL import Image
import cv2
import numpy as np

def generate_blocky_noise(width, height, block_size):
    """
    Generates a blocky noise image with exact requested dimensions.

    Args:
        width: Width of the image.
        height: Height of the image.
        block_size: Size of each block.

    Returns:
        A PIL Image object with exact (height, width) dimensions.
    """
    # Calculate grid dimensions, rounding UP to cover entire image
    grid_width = (width + block_size - 1) // block_size
    grid_height = (height + block_size - 1) // block_size

    # Generate random block colors
    random_colors = np.random.randint(0, 256, size=(grid_height, grid_width, 3), dtype=np.uint8)

    # Create full-size image with Kronecker product
    img_array = np.kron(random_colors, np.ones((block_size, block_size, 1), dtype=np.uint8))

    # Crop to exact requested dimensions
    return Image.fromarray(img_array[:height, :width])


# Generate and display the image
ima = generate_blocky_noise(2000, 500, 12)
plt.imshow(ima)
plt.title("Blocky Noise")
plt.axis('off')
plt.show()

# Convert to OpenCV BGR format
open_cv_image = np.array(ima)
open_cv_image = open_cv_image[:, :, ::-1]  # RGB to BGR

# Convert to grayscale
gray = cv2.cvtColor(open_cv_image, cv2.COLOR_BGR2GRAY)

# Apply Otsu's thresholding
_, thresh1 = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# Display thresholded image
plt.imshow(thresh1, cmap='gray')
plt.title("Otsu Thresholding")
plt.axis('off')
plt.show()

print(thresh1.shape)

In [ ]:
import  image_process_tool_box as ipt
import os
import cv2

reference_img = r"data\speckle_pattern_img\reference_im"
name = f"0_Generated_spec_image.tif"

image = cv2.imread(r"C:\Users\General User\nokop\pattern2\data\speckle_pattern_img\reference_im\0_Generated_spec_image.tif", cv2.IMREAD_GRAYSCALE)
print(image.shape)

column_to_remove = 250
new_image = np.delete(image, column_to_remove, axis=0)

print(new_image.shape)

cv2.imwrite(os.path.join(reference_img,name), new_image)

### Quick DIC

In [ ]:
import  image_process_tool_box as ipt
import os
import cv2

reference_img = r"data\speckle_pattern_img\reference_im\Validation"

listt = [1,2,3,4,5]
print(listt)

for i, pixsize in enumerate(listt):

    name = f"{240 + i*2}_Generated_spec_image.tif"

    # Create image
    ima = np.array(generate_blocky_noise(2000, 500, pixsize))
    plt.imshow(ima)
    plt.axis('off')
    plt.show()

    print(ima.shape)
    gray = cv2.cvtColor(ima, cv2.COLOR_BGR2GRAY)

    # Apply Otsu's thresholding
    _, thresh1 = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    save_path = os.path.join(reference_img, name)

    # Display thresholded image
    plt.imshow(thresh1, cmap='gray')
    plt.axis('off')
    plt.show()

    cv2.imwrite(save_path, thresh1)
    # deform



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

def update(frame, img, U, V, Du, Dv, feed, kill, dt):
    Lu = (
        np.roll(U, 1, axis=0)
        + np.roll(U, -1, axis=0)
        + np.roll(U, 1, axis=1)
        + np.roll(U, -1, axis=1)
        - 4 * U
    )
    Lv = (
        np.roll(V, 1, axis=0)
        + np.roll(V, -1, axis=0)
        + np.roll(V, 1, axis=1)
        + np.roll(V, -1, axis=1)
        - 4 * V
    )
    
    dUdt = Du * Lu - U * V**2 + feed * (1 - U)
    dVdt = Dv * Lv + U * V**2 - (kill + feed) * V
    
    U_new = U + dUdt * dt
    V_new = V + dVdt * dt
    
    U[:] = U_new
    V[:] = V_new
    
    img.set_array(V)
    return img,

grid_size = 256
Du, Dv = 0.16, 0.08
feed, kill = 0.035, 0.065
dt = 1.0
timesteps = 1000

U = np.random.rand(grid_size, grid_size)
V = np.random.rand(grid_size, grid_size)

fig, ax = plt.subplots()
img = ax.imshow(V, cmap='viridis', animated=True)

ani = animation.FuncAnimation(
    fig,
    update,
    fargs=(img, U, V, Du, Dv, feed, kill, dt),
    interval=50,
    blit=True,
    save_count=timesteps,
)

plt.show()

# 3. Interpolation and optimisation

In [ ]:
# Griddata implementation

import numpy as np
from scipy.interpolate import griddata

# Data points coordinates
points = np.array([[1,1],[1,2],[2,2],[2,1]]) # Coordinate array

# Vectors making up coordinate array
points_x = np.array([1,1,2,2]) 
points_y = np.array([1,2,2,1])
print('Points shape: ', points.shape)

# Data values at the respective coordinates
values = np.array([1,4,5,2])
print('Values shape: ', values.shape)

# Points at which to interpolate
xi=([1.2,1.2])
result=griddata(points, values, xi,  method='linear')
print("Value of interpolated function at",xi,"=",*result)


In [ ]:
import os
import image_process_tool_box as imgan

batch = 6
sundic_binary_folder    = rf"output\sundic_bin\batch_{batch}"

sundic_binary_files = sorted(
    [f for f in os.listdir(sundic_binary_folder)
    if f.endswith('results.sdic')],
    key = lambda x: int(x.split('_')[0])
)
print('\nSundic binary files number:', len(sundic_binary_files))


all_expected_prefixesbin = imgan.expected_prefixes(sundic_binary_folder)
prefix_positionsbin = {prefix: i for i, prefix in enumerate(all_expected_prefixesbin)}

all_expected_prefixesbin = [0,2,4]

for prefix_num in all_expected_prefixesbin:

    m = prefix_positionsbin[prefix_num] + 2

    print(f'\nCurrent prefix number is {prefix_num}')
    print(f'Current row number is {m}')

    sunfile = f'{prefix_num}_results.sdic'

    # sundic_data_path = os.path.join(sundic_save, sunfile)
    sundic_data_path = os.path.join(sundic_binary_folder, sunfile)

    print(f'Reading DIC data: {sundic_data_path}')

    try:

        if not os.path.exists(sundic_data_path):
            raise Exception("Sundic.sdic file not found")
        
        print("File found")

    except Exception as e:

        print(e)

In [ ]:
# Testing and evaluating different surrogate models

from scipy.interpolate import RBFInterpolator
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import image_process_tool_box as ipt

patt_class = 6

excel_path = rf'output\excel_docs\excel_{patt_class}'
p_metrics, meas_error, p_param, nans, indicators = ipt.read_spec_excel(
    excel_path, doc_num=3
)

metric_strings = (
                    "Mean subset fluctuation (MSF)", 
                    "Mean intensity gradient (MIG)", 
                    "E_f", 
                    "Mean intensity of the second derivative (MIOSD)",
                    "Shannon entropy", 
                    "Power area", 
                    "SSSIG", 
                    "Autocorrelation peak radius"
                )

# Prepare data
x_1 = p_param[:, 0]
x_2 = p_param[:, 1]
x_3 = p_param[:, 2]
x_4 = p_param[:, 3]
# x_5 = p_param[:, 4]
coordinate_arr = np.vstack((x_1, x_2, x_3, x_4)).T

best_poly_rscore = np.zeros((8))
mean_rscore = [-np.inf, None]

for oply in range(10):
    for column_number in  range(8):
        # z = meas_error[:, 0]
        z = p_metrics[:, column_number]

        # Split data
        X_train, X_test, z_train, z_test = train_test_split(
            coordinate_arr, z, test_size=0.2, random_state=42
        )

        # ============================================================================
        # MODEL 1: POLYNOMIAL REGRESSION
        # ============================================================================
        poly = PolynomialFeatures(degree=oply)  
        X_poly_train = poly.fit_transform(X_train)
        X_poly_test = poly.transform(X_test)
        poly_model = LinearRegression()
        poly_model.fit(X_poly_train, z_train)
        poly_pred = poly_model.predict(X_poly_test)
        
        poly_r2 = r2_score(z_test, poly_pred)

        best_poly_rscore[column_number] = poly_r2

        # ============================================================================
        # MODEL 2: RBF (RADIAL BASIS FUNCTIONS)
        # ============================================================================
        rbf_model = RBFInterpolator(
            X_train, z_train, kernel='thin_plate_spline', smoothing=0.1
        )
        rbf_pred = rbf_model(X_test)
        rbf_r2 = r2_score(z_test, rbf_pred)

        # ============================================================================
        # RESULTS
        # ============================================================================
        # print("Model Performance (R² scores):")
        # print(f"Polynomial (degree {ply}): {poly_r2:.4f}")
        # print(f"RBF:                   {rbf_r2:.4f}")

        # # Choose best model
        # scores = {'Polynomial': poly_r2, 'RBF': rbf_r2}
        # best_model = max(scores, key=scores.get)
        # print(f"\nBest model: {best_model} (R² = {scores[best_model]:.4f})")

        # # Plot comparison
        # fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

        # ax1.scatter(z_test, poly_pred, alpha=0.6)
        # ax1.plot([z_test.min(), z_test.max()], [z_test.min(), z_test.max()], 'r--')
        # ax1.set_title(f'Polynomial (R² = {poly_r2:.3f})')
        # ax1.set_xlabel(f'Actual {metric_strings[column_number]}')
        # ax1.set_ylabel(f'Predicted {metric_strings[column_number]}')
        # ax1.grid(True, alpha=0.3)

        # ax2.scatter(z_test, rbf_pred, alpha=0.6)
        # ax2.plot([z_test.min(), z_test.max()], [z_test.min(), z_test.max()], 'r--')
        # ax2.set_title(f'RBF (R² = {rbf_r2:.3f})')
        # ax2.set_xlabel(f'Actual {metric_strings[column_number]}')
        # ax2.set_ylabel(f'Predicted {metric_strings[column_number]}')
        # ax2.grid(True, alpha=0.3)

        # plt.tight_layout()
        # plt.show()

        # Plot the worst model - look at after inner loop, find lowest value in best_rscore and plot corresponding

    # Done with all 8 metrics for this degree
    new_mean_rscore = [np.mean(best_poly_rscore), oply]

    print(f'Polynomial order {oply}\n mean R² = {new_mean_rscore[0]:.4f}')

    if new_mean_rscore[0] > mean_rscore[0]:
        mean_rscore = new_mean_rscore

print(f'\nBest polynomial order is {mean_rscore[1]}')
print(f'Mean R² = {mean_rscore[0]:.4f}')


In [ ]:
import matplotlib.pyplot as plt
import numpy
from pyDOE import lhs

plt.style.use(['science', 'no-latex','ieee', 'grid'])

plt.rcParams.update({
    'font.size': 2,
    'axes.labelsize': 2,
    'xtick.labelsize': 2,
    'ytick.labelsize': 2,
    'legend.fontsize': 4,
    'lines.linewidth': 0.5,
    'lines.markersize': 2
})

l = lhs(2,20, criterion='center',iterations=100) # Latin Hypercube Sampling of two variables, and 10 samples each.

fig = plt.figure()
ax = fig.gca()
ax.set_xticks(numpy.arange(0,1,0.1))
ax.set_yticks(numpy.arange(0,1,0.1))
plt.scatter(l[:, 0], l[:, 1], color="r", label="LHS")
plt.grid(True)
plt.show()

#### Comparing error results that were calculated using different methods

In [ ]:
import matplotlib.pyplot as plt
import image_process_tool_box as ipt
import  numpy as np
import os
from scipy.interpolate import RBFInterpolator
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import image_process_tool_box as ipt
from scipy import stats
import numpy as np

# Read excel sheet 1
optimised_patterns_data = r'output\excel_docs'
_, omeas_error, _, _, _ = ipt.read_spec_excel(optimised_patterns_data, doc_num = 80)

# Get top 5 RMSE, bias and std. dev error
column = 2
mask = (omeas_error[:,4] != 0)

error = np.abs(omeas_error[:,column])

# Read excel sheet 2
optimised_patterns_data = r'output\excel_docs'
_, meas_error, _, _, _ = ipt.read_spec_excel(optimised_patterns_data, doc_num = 81)

mask2 = (meas_error[:,5] != 0)

error2 = np.abs(meas_error[:,column])

plt.scatter(error,error2)
plt.xlabel("RMSE from sqrt(mean(error_grid^2))")
plt.ylabel("RMSE from equation 6.10")
plt.grid(True)
plt.show()

In [ ]:
import numpy as np

# Length
n = 100  

# True values: sine wave
x = np.linspace(0, 2*np.pi, n)
true_values = np.sin(x)

# Measured values: sine wave + Gaussian noise
np.random.seed(0)  # reproducibility
measured_values = true_values + 0.1 * np.random.randn(n)

# Example RMSE calculation
rmse = np.sqrt(np.mean((measured_values - true_values)**2))

# print("True values:", true_values)
# print("Measured values:", measured_values)
print("RMSE:", rmse)

bias = np.mean(measured_values - true_values)
variance_error = np.std(measured_values-true_values)
 
rmse2 = np.sqrt(bias**2 + variance_error**2)
print("RMSE2:", rmse2)



#### Decision trees

In [6]:
from sklearn import tree
from scipy.interpolate import RBFInterpolator
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import image_process_tool_box as ipt

patt_class = [0,1,2,3,4,5,6,7,8,9,10,11]

metric_strings = (
                    "MSF", 
                    "MIG", 
                    "$E_f",
                    "MIOSD",
                    "Shannon entropy", 
                    "PSA", 
                    "SSSIG", 
                    "$R_{peak}$"
                )

score_per_metric = np.full((len(metric_strings),len(patt_class)), np.nan)

for class_index, pattern_class in enumerate(patt_class):

    if pattern_class in [1,7]:
        continue
        
    excel_path = rf'output\excel_docs\excel_{pattern_class}'
    p_metrics, meas_error, p_param, nans, indicators = ipt.read_spec_excel(
        excel_path, doc_num=None
    )

    errors = meas_error[:, 0]
    mask = ((~np.isnan(errors)) 
            & (errors != 0) 
            & (np.abs(errors - np.nanmean(errors)) <= 3 * np.nanstd(errors))
            )
 
    for metric_index in range(len(metric_strings)):

        x = p_metrics[mask,metric_index]
        y = meas_error[mask,0]

        r = np.corrcoef(x, y)

        score_per_metric[metric_index,class_index] = np.round(r[0,1],2)

print("\n\n-----------------------------------------")
print("Score per metric (indicated by row) across pattern classes (columns)")
print(score_per_metric)

average_score_per_metric = np.nanmean(score_per_metric,axis=1)

print("\n\n-----------------------------------------")
print("Average score per metric across classes")
print(f"{np.round(average_score_per_metric,2)}")

# print("\n\n-----------------------------------------")
# print("Average score")
# print(f"{np.mean(np.round(average_score_per_metric,2))}")


Current doc_number (Error analysis): 12
Current doc_number (Error analysis): 14
Current doc_number (Error analysis): 14
Current doc_number (Error analysis): 13
Current doc_number (Error analysis): 13
Current doc_number (Error analysis): 14
Current doc_number (Error analysis): 15
Current doc_number (Error analysis): 13
Current doc_number (Error analysis): 13
Current doc_number (Error analysis): 16


-----------------------------------------
Score per metric (indicated by row) across pattern classes (columns)
[[-0.86   nan -0.68 -0.81 -0.97 -0.96 -0.85   nan -0.78 -0.35  0.31 -0.47]
 [-0.82   nan -0.67 -0.64 -0.93 -0.85 -0.85   nan -0.53 -0.61 -0.94 -0.47]
 [-0.39   nan -0.2  -0.37 -0.78 -0.6  -0.84   nan -0.28 -0.28  0.17 -0.56]
 [-0.81   nan -0.66 -0.64 -0.92 -0.83 -0.84   nan -0.41 -0.62 -0.93 -0.43]
 [-0.81   nan  0.35  0.35 -0.97 -0.57 -0.8    nan -0.18  0.11 -0.42  0.26]
 [ 0.91   nan  0.3   0.33  0.9   0.87  0.91   nan  0.19  0.53  0.86  0.39]
 [-0.82   nan -0.67 -0.64 -0.93 -0.85

In [ ]:
# Scores after data adjustment

from sklearn import tree
from scipy.interpolate import RBFInterpolator
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import image_process_tool_box as ipt


correction_model = [
    'none',
    'sinusoidal',
    'cubic',
    'perlin_blobs'
]
mod = 3

print(f"Model: {correction_model[mod]}\n--------------")

patt_class = [0,1,2,3,4,5,6,7,8,9,10,11]

metric_strings = (
                    "MSF", 
                    "MIG", 
                    "$E_f",
                    "MIOSD",
                    "Shannon entropy", 
                    "PSA", 
                    "SSSIG", 
                    "$R_{peak}$"
                )

score_per_metric = np.full((len(metric_strings),len(patt_class)), np.nan)

for class_index, pattern_class in enumerate(patt_class):

    if pattern_class in [0,1,2,3,4,5,6,7,8,9,10]:
        continue
        
    excel_path = rf"output\excel_docs\after_corr_{correction_model[mod]}"
    p_metrics, meas_error, p_param, nans, indicators = ipt.read_spec_excel(
        excel_path, doc_name = "corrected_batch", doc_num=pattern_class,
    )

    errors = meas_error[:, 0]
    mask = ((~np.isnan(errors)) 
            & (errors != 0) 
            & (np.abs(errors - np.nanmean(errors)) <= 3 * np.nanstd(errors))
            )
 
    for metric_index in range(len(metric_strings)):

        x = p_metrics[mask,metric_index]
        y = meas_error[mask,0]

        r = np.corrcoef(x, y)

        score_per_metric[metric_index,class_index] = np.round(r[0,1],2)

print("\n\n-----------------------------------------")
print("Score per metric (indicated by row) across pattern classes (columns)")
print(score_per_metric)

average_score_per_metric = np.nanmean(score_per_metric,axis=1)

print("\n\n-----------------------------------------")
print("Average score per metric across classes")
print(f"{np.round(average_score_per_metric,2)}")

print("\n\n-----------------------------------------")
print("Average score")
print(f"{np.mean(np.round(average_score_per_metric,2))}")


Model: perlin_blobs
--------------
Current doc_number (Error analysis): 12


FileNotFoundError: [Errno 2] No such file or directory: 'output\\excel_docs\\after_corr_perlin_blobs\\corrected_batch_12.xlsx'

In [3]:
# Spearman coefficient

from sklearn import tree
from scipy.interpolate import RBFInterpolator
from scipy.stats import spearmanr  # Add this import
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import image_process_tool_box as ipt

patt_class = [0,1,2,3,4,5,6,7,8,9,10,11]

metric_strings = (
                    "MSF", 
                    "MIG", 
                    "$E_f",
                    "MIOSD",
                    "Shannon entropy", 
                    "PSA", 
                    "SSSIG", 
                    "$R_{peak}$"
                )

score_per_metric = np.full((len(metric_strings),len(patt_class)), np.nan)

for class_index, pattern_class in enumerate(patt_class):

    if pattern_class == 1 or pattern_class == 7:
        continue
        
    excel_path = rf'output\excel_docs\excel_{pattern_class}'
    p_metrics, meas_error, p_param, nans, indicators = ipt.read_spec_excel(
        excel_path, doc_num=None
    )

    errors = meas_error[:, 0]
    mask = (~np.isnan(errors)) & (errors != 0) & (np.abs(errors - np.nanmean(errors)) <= 3 * np.nanstd(errors))
 
    for metric_index in range(len(metric_strings)):

        x = p_metrics[mask,metric_index]
        y = meas_error[mask,0]

        # Spearman's rank correlation
        rho, p_value = spearmanr(x, y)

        score_per_metric[metric_index,class_index] = np.round(rho, 2)

print("\n\n-----------------------------------------")
print(score_per_metric)

average_score_per_metric = np.nanmean(score_per_metric,axis=1)

print("\n\n-----------------------------------------")
print(f"{np.round(average_score_per_metric,2)}")

Current doc_number (Error analysis): 12
Current doc_number (Error analysis): 14
Current doc_number (Error analysis): 14
Current doc_number (Error analysis): 13
Current doc_number (Error analysis): 13
Current doc_number (Error analysis): 14
Current doc_number (Error analysis): 15
Current doc_number (Error analysis): 13
Current doc_number (Error analysis): 13
Current doc_number (Error analysis): 16


-----------------------------------------
[[-0.93   nan -0.24 -0.38 -0.51 -0.68 -0.88   nan -0.95 -0.85  0.62 -0.37]
 [-0.94   nan  0.04  0.07 -0.39 -0.43 -0.89   nan -0.31 -0.52 -0.69  0.39]
 [-0.37   nan  0.17  0.14 -0.2  -0.26 -0.9    nan  0.25 -0.59  0.4   0.45]
 [-0.94   nan  0.04  0.07 -0.34 -0.42 -0.91   nan -0.17 -0.41 -0.58  0.41]
 [-0.82   nan  0.03  0.27 -0.47 -0.15 -0.82   nan -0.45 -0.48 -0.44 -0.03]
 [ 0.96   nan -0.08 -0.16  0.47  0.51  0.96   nan  0.03 -0.01  0.62 -0.27]
 [-0.94   nan  0.04  0.08 -0.39 -0.42 -0.89   nan -0.26 -0.49 -0.76  0.4 ]
 [ 0.85   nan -0.04 -0.1   0.27

#### Polynomial regression

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import image_process_tool_box as ipt
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error 
from sklearn.exceptions import DataConversionWarning
from sklearn.model_selection import ParameterGrid
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

import warnings
warnings.filterwarnings(action='ignore', category=DataConversionWarning)

pattern_class_names = (
    "PT_Speckles",
    "PT_Checkerboard",
    "PT_Perlin",
    "PT_Perlin cube",
    "PT_Perlin bandlimit",
    "PT_Perlin sin",
    
    "Speckles",
    "Checkerboard",
    "Perlin",
    "Perlin cube",
    "Perlin bandlimit",
    "Perlin sin",
    
)


# Get data
keeping_average_R2 = np.zeros((12,1))
metric_wise_average = np.full((12,8),np.nan)

for patt_class in range(12):

    # Read data
    print("\n-----------------\n", pattern_class_names[patt_class],"\n-----------------")
    excel_path = rf'output\excel_docs\excel_{patt_class}'
    p_metrics, meas_error, p_param, nans, indicators = ipt.read_spec_excel(
        excel_path, doc_num=None
    )

    # Columns 
    metric_strings = (
                        "MSF", 
                        "MIG", 
                        "E_f", 
                        "MIOSD",
                        "Shannon", 
                        "Power", 
                        "SSSIG", 
                        "Peak radius"
                    )

    error_strings = (
        "RMS error",
        "Bias error",
        "Variance error"
    )

    errors = meas_error[:, 0]
    mask = (~np.isnan(errors)) & (errors != 0) & (np.abs(errors - np.nanmean(errors)) <= 3 * np.nanstd(errors))
 

    x_1 = p_param[mask, 0]
    x_2 = p_param[mask, 1]
    x_3 = p_param[mask, 2]
    x_4 = p_param[mask, 3]
    x_5 = p_param[mask, 4]

    # x_1 = p_param[:, 0]
    # x_2 = p_param[:, 1]
    # x_3 = p_param[:, 2]
    # x_4 = p_param[:, 3]
    # x_5 = p_param[:, 4]
    print(f"\nData count = {len(x_1)}")

    if patt_class == 0 or patt_class == 6:
        coordinate_arr = np.vstack((x_1, x_3, x_4, x_5)).T          # Speckles variables
    elif patt_class == 1 or patt_class == 7:
        coordinate_arr = x_1.T                                      # Checkerboard variables
        coordinate_arr = coordinate_arr.reshape(-1, 1)
    else:
        coordinate_arr = np.vstack((x_1, x_2, x_3, x_4)).T          # Perlin noise variables

    ran = len(p_metrics[0,:])
    r_score_arr = np.full((1, ran), np.nan)
    Modelling_error = False

    for iii in range(ran):
        # coordinate_arr = p_metrics[mask,iii].reshape(-1,1)

        # z = np.abs(meas_error[mask, 0].reshape(-1, 1))
        if Modelling_error:
            z = np.abs(meas_error[mask, 0].reshape(-1, 1))
        else:
            z = p_metrics[mask, iii].reshape(-1, 1)


        X_train, X_test, z_train, z_test = train_test_split(
            coordinate_arr, z, test_size=0.2, random_state=42
        )

        # Scale X only
        feature_scaler = StandardScaler()
        X_train_scaled = feature_scaler.fit_transform(X_train)
        X_test_scaled = feature_scaler.transform(X_test)

        # Create pipeline
        poly_pipeline = Pipeline([
            ('poly', PolynomialFeatures()),
            ('linear', LinearRegression())
        ])

        # Parameter grid for GridSearchCV
        param_grid = {
            'poly__degree': [1, 2, 3, 4],
        }

        # GridSearchCV
        grid = GridSearchCV(
            poly_pipeline,
            param_grid=param_grid,
            cv=5,
            scoring='r2',
            n_jobs=-1
        )

        # Fit and predict
        grid.fit(X_train_scaled, z_train.ravel())
        test_results = grid.predict(X_test_scaled)

        # Evaluate
        r_score_arr[0, iii] = np.round(r2_score(z_test, test_results), 3)
    
    # Store metric-wise scores
    metric_wise_average[patt_class, :] = r_score_arr[0, :]
    
    if patt_class == 1 or patt_class == 7:
        keeping_average_R2[patt_class] = np.nan
    else:
        keeping_average_R2[patt_class] = np.round(np.mean(r_score_arr),3)
        
    print('\nR^2 score table\n')
    print(metric_strings)
    print(r_score_arr)
    # print(f"\nAverage = {np.round(np.mean(r_score_arr),3)}")

print(f'\nAverage model score: {np.nanmean(keeping_average_R2):.3f}')
print(f'\nAverage per metric (down columns):')
for i, metric in enumerate(metric_strings):
    print(f'  {metric}: {np.nanmean(metric_wise_average[:, i]):.3f}')

#### Gaussian processes

In [ ]:
# https://distill.pub/2019/visual-exploration-gaussian-processes/

# Gaussian process regression with gridsearch optimisation

import numpy as np
import matplotlib.pyplot as plt
import image_process_tool_box as ipt
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import DotProduct, WhiteKernel,RBF, Matern, ExpSineSquared, RationalQuadratic,ConstantKernel
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
from sklearn.exceptions import DataConversionWarning, ConvergenceWarning,FitFailedWarning
from sklearn.model_selection import ParameterGrid

import warnings
warnings.filterwarnings(action='ignore', category=DataConversionWarning)
warnings.filterwarnings(action='ignore', category=ConvergenceWarning)
warnings.filterwarnings(action='ignore', category=FitFailedWarning)
warnings.filterwarnings(action='ignore', category=UserWarning)

pattern_class_names = (
    "PT_Speckles",
    "PT_Checkerboard",
    "PT_Perlin",
    "PT_Perlin cube",
    "PT_Perlin bandlimit",
    "PT_Perlin sin",

    "Speckles",
    "Checkerboard",
    "Perlin",
    "Perlin cube",
    "Perlin bandlimit",
    "Perlin sin",
)



# Get data
keeping_average_R2 = np.zeros((12,1))
metric_wise_average = np.full((12,8),np.nan)

for patt_class in range(12):

    # Read data
    print("\n-----------------\n", pattern_class_names[patt_class],"\n-----------------")
    excel_path = rf'output\excel_docs\excel_{patt_class}'
    p_metrics, meas_error, p_param, nans, indicators = ipt.read_spec_excel(
        excel_path, doc_num=None
    )

    # Columns
    metric_strings = (
                        "MSF",
                        "MIG",
                        "E_f",
                        "MIOSD",
                        "Shannon",
                        "Power",
                        "SSSIG",
                        "Peak radius"
                    )

    error_strings = (
        "RMS error",
        "Bias error",
        "Variance error"
    )

    errors = meas_error[:, 0]
    mask = (~np.isnan(errors)) & (errors != 0) & (np.abs(errors - np.nanmean(errors)) <= 3 * np.nanstd(errors))


    x_1 = p_param[mask, 0]
    x_2 = p_param[mask, 1]
    x_3 = p_param[mask, 2]
    x_4 = p_param[mask, 3]
    x_5 = p_param[mask, 4]

    # x_1 = p_param[:, 0]
    # x_2 = p_param[:, 1]
    # x_3 = p_param[:, 2]
    # x_4 = p_param[:, 3]
    # x_5 = p_param[:, 4]
    print(f"\nData count = {len(x_1)}")

    if patt_class == 0 or patt_class == 6:
        coordinate_arr = np.vstack((x_1, x_3, x_4, x_5)).T          # Speckles variables
    elif patt_class == 1 or patt_class == 7:
        coordinate_arr = x_1.T                                      # Checkerboard variables
        coordinate_arr = coordinate_arr.reshape(-1, 1)
    else:
        coordinate_arr = np.vstack((x_1, x_2, x_3, x_4)).T          # Perlin noise variables

    ran = len(p_metrics[0,:])
    r_score_arr = np.full((1, ran), np.nan)

    modelling_average=np.full((1,12), np.nan)
    Modelling_error = True

    for iii in range(ran):

        coordinate_arr = p_metrics[mask,iii].reshape(-1,1)

        if Modelling_error:
            z = np.abs(meas_error[mask, 0].reshape(-1, 1))
        else:
            z = p_metrics[mask, iii].reshape(-1, 1)

        # Split data
        X_train, X_test, z_train, z_test = train_test_split(
            coordinate_arr, z, test_size=0.2, random_state=42
        )

        # Scale features put them all in the same range
        scaler = StandardScaler()                                   # Expects 2D array
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        z_train_scaled = scaler.fit_transform(z_train)
        z_test_scaled = scaler.transform(z_test)


        # ==============GridsearchCV====================
        meta_model = GaussianProcessRegressor()

        p_grid = {
            'kernel': [
                ConstantKernel(),                 # constant
                DotProduct(),                     # linear
                DotProduct(sigma_0=1.0)**3,      # polynomial
                RBF(),                           # squared exponential
                Matern(nu=1.5),                  # matern
                Matern(nu=0.5),                  # exponential
                ExpSineSquared(),                # gamma-exponential (periodic)
                RationalQuadratic(),             # rational quadratic
            ],
            'alpha': np.logspace(-10, -1, 10)
        }
        grid = GridSearchCV(
            meta_model,
            param_grid=p_grid,
            cv = 5,
            n_jobs=-1,
            scoring='r2')

        # ==============================================
        # Train
        grid.fit(X_train_scaled, z_train_scaled.ravel())

        # Evaluate on test data
        test_results = grid.predict(X_test_scaled)
        r_score_arr[0,iii] = np.round(r2_score(z_test_scaled, test_results), 3)
        # r_score_arr[0,iii] = np.round(grid.score(X_test_scaled,z_test_scaled), 3)
    metric_wise_average[patt_class, :] = r_score_arr[0, :]
    
    if patt_class == 1 or patt_class == 7:
        keeping_average_R2[patt_class] = np.nan
    else:
        keeping_average_R2[patt_class] = np.round(np.mean(r_score_arr),3)
        
    print('\nR^2 score table\n')
    print(metric_strings)
    print(r_score_arr)
    # print(f"\nAverage = {np.round(np.mean(r_score_arr),3)}")

print(f'\nAverage model score: {np.nanmean(keeping_average_R2):.3f}')
print(f'\nAverage per metric (down columns):')
for i, metric in enumerate(metric_strings):
    print(f'  {metric}: {np.nanmean(metric_wise_average[:, i]):.3f}')

#### Support vector regression (Metric evaluation)

In [ ]:
# Support vector regression with gridsearch optimisation
 
# Support vector regression (SVR)
import numpy as np
import matplotlib.pyplot as plt
import image_process_tool_box as ipt
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error 
from sklearn.exceptions import DataConversionWarning
from sklearn.model_selection import ParameterGrid

import warnings
warnings.filterwarnings(action='ignore', category=DataConversionWarning)

pattern_class_names = (
    "PT_Speckles",
    "PT_Checkerboard",
    "PT_Perlin",
    "PT_Perlin cube",
    "PT_Perlin bandlimit",
    "PT_Perlin sin",
    
    "Speckles",
    "Checkerboard",
    "Perlin",
    "Perlin cube",
    "Perlin bandlimit",
    "Perlin sin",
    
)


# Get data
keeping_average_R2 = np.zeros((12,1))

for patt_class in range(12):

    # Read data
    print("\n-----------------\n", pattern_class_names[patt_class],"\n-----------------")
    excel_path = rf'output\excel_docs\excel_{patt_class}'
    p_metrics, meas_error, p_param, nans, indicators = ipt.read_spec_excel(
        excel_path, doc_num=None
    )

    # Columns 
    metric_strings = (
                        "MSF", 
                        "MIG", 
                        "E_f", 
                        "MIOSD",
                        "Shannon", 
                        "Power", 
                        "SSSIG", 
                        "Peak radius"
                    )

    error_strings = (
        "RMS error",
        "Bias error",
        "Variance error"
    )

    errors = meas_error[:, 0]
    mask = (~np.isnan(errors)) & (errors != 0) & (np.abs(errors - np.nanmean(errors)) <= 3 * np.nanstd(errors))


    x_1 = p_param[mask, 0]
    x_2 = p_param[mask, 1]
    x_3 = p_param[mask, 2]
    x_4 = p_param[mask, 3]
    x_5 = p_param[mask, 4]

    # x_1 = p_param[:, 0]
    # x_2 = p_param[:, 1]
    # x_3 = p_param[:, 2]
    # x_4 = p_param[:, 3]
    # x_5 = p_param[:, 4]
    print(f"\nData count = {len(x_1)}")

    if patt_class == 0 or patt_class == 6:
        coordinate_arr = np.vstack((x_1, x_3, x_4, x_5)).T          # Speckles variables
    elif patt_class == 1 or patt_class == 7:
        coordinate_arr = x_1.T                                      # Checkerboard variables
        coordinate_arr = coordinate_arr.reshape(-1, 1)
    else:
        coordinate_arr = np.vstack((x_1, x_2, x_3, x_4)).T          # Perlin noise variables

    ran = len(p_metrics[0,:])
    r_score_arr = np.full((1, ran), np.nan)

    Modelling_error = False

    for iii in range(ran):

        # coordinate_arr = p_metrics[mask,iii].reshape(-1,1)

        if Modelling_error:
            z = np.abs(meas_error[mask, 0].reshape(-1, 1))
        else:
            z = p_metrics[mask, iii].reshape(-1, 1)


        # Split data
        X_train, X_test, z_train, z_test = train_test_split(
            coordinate_arr, z, test_size=0.2, random_state=42
        )

        # Scale features put them all in the same range
        scaler = StandardScaler()                                   # Expects 2D array
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)        
        z_train_scaled = scaler.fit_transform(z_train)
        z_test_scaled = scaler.transform(z_test)


        # ==============GridsearchCV====================
        meta_model = SVR()                
        
        p_grid = [
            # RBF kernel
            {
                'kernel': ['rbf'],
                'C': np.logspace(-3, 3, 7),
                'gamma': ['scale', 'auto'],
            },
            # Polynomial kernel
            # {
            #     'kernel': ['poly'],
            #     'C': np.logspace(-3, 3, 7),
            #     'degree': [2, 3, 4],
            #     'gamma': ['scale'],
            # },
            # # # Sigmoid kernel
            # {
            #     'kernel': ['sigmoid'],
            #     'C': np.logspace(-3, 3, 7),
            #     'gamma': ['scale'],
            # }
        ]

        grid = GridSearchCV(
            meta_model, 
            param_grid=p_grid, 
            cv = 5, 
            n_jobs=-1,
            scoring='r2')

        # ==============================================
        # Train
        grid.fit(X_train_scaled, z_train_scaled)

        # Evaluate on test data
        test_results = grid.predict(X_test_scaled)

        r_score_arr[0,iii] = np.round(r2_score(z_test_scaled, test_results), 3)
        # r_score_arr[0,iii] = np.round(grid.score(X_test_scaled,z_test_scaled), 3)

    # Ignore checkerboard pattern classes when calculating average
    if patt_class == 1 or patt_class == 7:
        
        keeping_average_R2[patt_class] = np.nan
    else:
        keeping_average_R2[patt_class] = np.round(np.mean(r_score_arr),3)
        

    print('\nR^2 score table\n')
    print(metric_strings)
    print(r_score_arr)
    # print(f"\nAverage = {np.round(np.mean(r_score_arr),3)}")

print(f'\nAverage model score: {np.nanmean(keeping_average_R2):.3f}')

In [ ]:
from sklearn import tree
from scipy.interpolate import RBFInterpolator
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_friedman2
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import DotProduct, WhiteKernel,RBF, Matern, ExpSineSquared, RationalQuadratic
from sklearn.gaussian_process.kernels import ConstantKernel as C, RationalQuadratic as RQ,ExpSineSquared as Exp
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import image_process_tool_box as ipt
from sklearn.preprocessing import StandardScaler
from sklearn.exceptions import ConvergenceWarning
import warnings
warnings.filterwarnings(action='ignore', category=ConvergenceWarning)

pattern_class_names = (
    "PT_Speckles",
    "PT_Checkerboard",
    "PT_Perlin",
    "PT_Perlin cube",
    "PT_Perlin bandlimit",
    "PT_Perlin sin",
    
    "Speckles",
    "Checkerboard",
    "Perlin",
    "Perlin cube",
    "Perlin bandlimit",
    "Perlin sin",
    
)

# Get data
keeping_average_R2 = np.zeros((12,1))

for patt_class in range(12):
    print("\n-----------------\n", pattern_class_names[patt_class],"\n-----------------")
    excel_path = rf'output\excel_docs\excel_{patt_class}'
    p_metrics, meas_error, p_param, nans, indicators = ipt.read_spec_excel(
        excel_path, doc_num=None
    )

    # Columns 
    metric_strings = (
                        "MSF", 
                        "MIG", 
                        "E_f", 
                        "MIOSD",
                        "Shannon", 
                        "Power", 
                        "SSSIG", 
                        "Peak radius"
                    )

    error_strings = (
        "RMS error",
        "Bias error",
        "Variance error"
    )

    errors = meas_error[:, 0]
    mask = (~np.isnan(errors)) & (errors != 0) & (np.abs(errors - np.nanmean(errors)) <= 3 * np.nanstd(errors))
 
    x_1 = p_param[mask, 0]
    x_2 = p_param[mask, 1]
    x_3 = p_param[mask, 2]
    x_4 = p_param[mask, 3]
    x_5 = p_param[mask, 4]
    print(f"\nData count = {len(x_1)}")

    if patt_class == 0 or patt_class == 6:
        coordinate_arr = np.vstack((x_1, x_3, x_4, x_5)).T          # Speckles variables
    elif patt_class == 1 or patt_class == 7:
        coordinate_arr = x_1.T                                      # Checkerboard variables
        coordinate_arr = coordinate_arr.reshape(-1, 1)
    else:
        coordinate_arr = np.vstack((x_1, x_2, x_3, x_4)).T          # Perlin noise variables

    ran = len(p_metrics[0,:])
    r_score_arr = np.full((1, ran), np.nan)
    best_kernel_arr = np.full((1, ran), "", dtype=object)

    # List of kernel combinations to try
    kernel_combinations = [
        ("RBF", RBF()),
        ("RBF + White Noise", RBF() + WhiteKernel()),
        ("Constant * RBF", C() * RBF()),
        ("Constant * RBF + White Noise", C() * RBF() + WhiteKernel()),
        
        ("Constant", C()),
        ("Linear", DotProduct()),
        ("Polynomial (degree=2)", DotProduct(sigma_0=1.0)**2),
        ("Polynomial (degree=3)", DotProduct(sigma_0=1.0)**3),
        ("Squared Exponential", RBF()),  # Note: RBF is the squared exponential kernel
        ("Matern (nu=0.5)", Matern(nu=0.5)),  # Exponential kernel
        ("Matern (nu=1.5)", Matern(nu=1.5)),
        ("Matern (nu=2.5)", Matern(nu=2.5)),
        ("Exponential", Matern(nu=0.5)),  # Exponential is Matern with nu=0.5
        ("Gamma-Exponential (Periodic)", ExpSineSquared()),
        ("Rational Quadratic", RationalQuadratic()),
        
    ]

    for iii in range(ran):
        # coordinate_arr = np.vstack((p_metrics[mask,7]))
        # z = np.abs(meas_error[mask, 0].reshape(-1, 1))
        z = p_metrics[mask, iii].reshape(-1, 1)

        # Split data
        X_train, X_test, z_train, z_test = train_test_split(
            coordinate_arr, z, test_size=0.2, random_state=42
        )

        # Scale features put them all in the same range
        scaler = StandardScaler()                                   # Expects 2D array
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)        
        z_train_scaled = scaler.fit_transform(z_train)
        z_test_scaled = scaler.transform(z_test)

        best_result = float('-inf')
        kernel_name = None
        best_model = None

        for name, kernel in kernel_combinations:
            gpr = GaussianProcessRegressor(kernel=kernel, random_state=0, n_restarts_optimizer=5)
            gpr.fit(X_train_scaled, z_train_scaled)
            
            test_r2 = gpr.score(X_test_scaled, z_test_scaled)
            
            if test_r2 > best_result:
                best_result = test_r2
                kernel_name = name
                best_model = gpr

        r_score_arr[0,iii] = np.round(best_result, 3)
        best_kernel_arr[0,iii] = kernel_name

    print('\nR^2 score table\n')
    print(metric_strings)
    print(r_score_arr)
    print(f"\nAverage = {np.round(np.mean(r_score_arr),3)}")
    
    print('\nBest kernel table\n')
    print(metric_strings)
    print(best_kernel_arr)

    keeping_average_R2[patt_class] = np.round(np.mean(r_score_arr),3)

print(f'\nAverage model score: {np.mean(keeping_average_R2):.3f}')

In [ ]:
# Support vector regression to evaluate metrics against RMSE and IQR respectively

# Support vector regression (SVR)
import numpy as np
import matplotlib.pyplot as plt
import image_process_tool_box as ipt
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error 
from sklearn.exceptions import DataConversionWarning
from sklearn.model_selection import ParameterGrid

import warnings
warnings.filterwarnings(action='ignore', category=DataConversionWarning)

pattern_class_names = (
    "PT_Speckles",
    "PT_Checkerboard",
    "PT_Perlin",
    "PT_Perlin cube",
    "PT_Perlin bandlimit",
    "PT_Perlin sin",
    
    "Speckles",
    "Checkerboard",
    "Perlin",
    "Perlin cube",
    "Perlin bandlimit",
    "Perlin sin",
    
)

# Get data
for patt_class in range(12):

    # Read data
    print("\n-----------------\n", pattern_class_names[patt_class],"\n-----------------")
    excel_path = rf'output\excel_docs\excel_{patt_class}'
    p_metrics, meas_error, p_param, nans, indicators = ipt.read_spec_excel(
        excel_path, doc_num=None
    )

    # Columns 
    metric_strings = (
                        "MSF", 
                        "MIG", 
                        "E_f", 
                        "MIOSD",
                        "Shannon", 
                        "Power", 
                        "SSSIG", 
                        "Peak radius"
                    )

    error_strings = (
        "RMS error",
        "Bias error",
        "Variance error"
    )

    errors = meas_error[:, 0]
    mask = (~np.isnan(errors)) & (errors != 0) & (np.abs(errors - np.nanmean(errors)) <= 3 * np.nanstd(errors))

    ran = len(p_metrics[0,:])
    r_score_arr = np.full((1, ran), np.nan)

    for iii in range(ran):
        # coordinate_arr = p_metrics[mask,iii].reshape(-1,1)
        print(f'coordarr shape:{coordinate_arr.shape}')
        z = np.abs(meas_error[mask, 0].reshape(-1, 1))
        print(f'Z shape:{z.shape}')

        # z = p_metrics[mask, iii].reshape(-1, 1)

        # Split data
        X_train, X_test, z_train, z_test = train_test_split(
            coordinate_arr, z, test_size=0.2, random_state=42
        )

        # Scale features put them all in the same range
        scaler = StandardScaler()                                   # Expects 2D array
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)        
        z_train_scaled = scaler.fit_transform(z_train)
        z_test_scaled = scaler.transform(z_test)


        # ==============GridsearchCV====================
        meta_model = SVR()                
        
        p_grid = [
            # RBF kernel
            {
                'kernel': ['rbf'],
                'C': np.logspace(-3, 3, 7),
                'gamma': ['scale', 'auto'],
            },
            # Polynomial kernel
            # {
            #     'kernel': ['poly'],
            #     'C': np.logspace(-3, 3, 7),
            #     'degree': [2, 3, 4],
            #     'gamma': ['scale'],
            # },
            # # Sigmoid kernel
            # {
            #     'kernel': ['sigmoid'],
            #     'C': np.logspace(-3, 3, 7),
            #     'gamma': ['scale'],
            # }
        ]

        grid = GridSearchCV(
            meta_model, 
            param_grid=p_grid, 
            cv = 5, 
            n_jobs=-1)

        # ==============================================
        # Train
        grid.fit(X_train_scaled, z_train_scaled)

        # Evaluate on test data
        test_results = grid.predict(X_test_scaled)

        r_score_arr[0,iii] = np.round(r2_score(z_test_scaled, test_results), 3)
        # r_score_arr[0,iii] = np.round(grid.score(X_test_scaled,z_test_scaled), 3)

    print('\nR^2 score table\n')
    print(metric_strings)
    print(r_score_arr)
    print(f"\nAverage = {np.round(np.mean(r_score_arr),3)}")


In [ ]:
# Relationship between parameters and error

from scipy.interpolate import griddata
from scipy.optimize import curve_fit
from pykrige.ok import OrdinaryKriging
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from mpl_toolkits.mplot3d import Axes3D
from sklearn.preprocessing import PolynomialFeatures
import image_process_tool_box as ipt

pattern_class_names = (
    "PT_Speckles",
    "PT_Checkerboard",
    "PT_Perlin",
    "PT_Perlin cube",
    "PT_Perlin sin",
    "PT_Perlin blobs",
    "Speckles",
    "Checkerboard",
    "Perlin",
    "Perlin cube",
    "Perlin sin",
    "Perlin blobs"
)

for classs in range(12):
    print("\n-----------------\n", pattern_class_names[classs],"\n-----------------")
    excel_path = rf'output\excel_docs\excel_{classs}'
    p_metrics,meas_error,p_param,nans,indicators = ipt.read_spec_excel(excel_path, doc_num=1)


    # Coordinate array
    x_1 = p_param[:, 0]
    x_2 = p_param[:, 1]
    x_3 = p_param[:, 2]
    x_4 = p_param[:, 3]
    x_5 = p_param[:, 4]
    x_6 = nans[:,0]


    coordinate_arr = np.vstack((x_1, x_2, x_3, x_4, x_5)).T
    z = meas_error[:, 0]

    try:
        fig, axs = plt.subplots(1, 5, figsize=(20, 4))

        axs[0].scatter(x_1, z)
        axs[0].set_title("Param 1")

        axs[1].scatter(x_2, z)
        axs[1].set_title("Param 2")

        axs[2].scatter(x_3, z)
        axs[2].set_title("Param 3")

        axs[3].scatter(x_4, z)
        axs[3].set_title("Param 4")

        axs[4].scatter(x_5, z)
        axs[4].set_title("Param 5")

        for ax in axs:
            ax.set_xlabel("Pattern parameter")
            ax.set_ylabel("z")
            ax.grid(True)

        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"Subplot plotting failed: {e}")


In [ ]:
import matplotlib.pyplot as plt
import image_process_tool_box as ipt
import  numpy as np
import os
from scipy.interpolate import RBFInterpolator
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import image_process_tool_box as ipt
from scipy import stats
import numpy as np

objective = 1

obj = objective - 1
batch = 6
image_prefix = obj * 24 + batch*2
excel_vector_index = int(image_prefix / 2)
print(f"Optimised image number = {image_prefix}")
print(f"excel index = {excel_vector_index}")

# Load optimised data
optimised_patterns_data = r'output\excel_docs'
op_metrics, omeas_error, op_param, onans, oindicators = ipt.read_spec_excel(optimised_patterns_data, doc_num = 59)

# Load batch data
excel_folder_address = rf'output\excel_docs\excel_{batch}'
p_metrics, meas_error, p_param, nans, indicators = ipt.read_spec_excel(excel_folder_address, doc_num=7)

mask = (meas_error[:,0] != 0) & (~np.isnan(meas_error[:,0])) & (np.abs(np.nanmean(meas_error[:,0]) - meas_error[:,0]) <= 3*np.std(meas_error[:,0]))

opt_error = np.min(omeas_error[excel_vector_index,0])
print(opt_error)

# Get percentile
percentile_in_sample = stats.percentileofscore(meas_error[mask,0], opt_error, kind='strict')
print(f"Percentile: {percentile_in_sample}")


In [ ]:
import matplotlib.pyplot as plt
import image_process_tool_box as ipt
import  numpy as np
import os
from scipy.interpolate import RBFInterpolator
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import image_process_tool_box as ipt
from scipy import stats
import numpy as np
import pandas as pd

analyse_error = True

if analyse_error:
    percentile_matrix = np.full((10,13), np.nan, dtype=object)
    percentile_matrix[1:,0] = ["RMSE","MSF","MIG","Ef","MIOSD","Shan","PSA","SSI","R"]
    values_matrix = np.full((10,13), np.nan, dtype=object)  # Create separate matrix
    values_matrix[1:,0] = ["RMSE","MSF","MIG","Ef","MIOSD","Shan","PSA","SSI","R"]
    objectives = [0,1,2,3,4,5,6,7,8]
else:
    percentile_matrix = np.full((9,13), np.nan, dtype=object)
    percentile_matrix[1:,0] = ["MSF","MIG","Ef","MIOSD","Shan","PSA","SSI","R"]
    objectives = [1,2,3,4,5,6,7,8]
    error_cols = [0,4,1]

percentile_matrix[0,1:] = ["C1","C2","C3","C4","C5","C6","C7","C8","C9","C10","C11","C12"]
values_matrix[0,1:] = ["C1","C2","C3","C4","C5","C6","C7","C8","C9","C10","C11","C12"]  # Add this line


for inde,obj in enumerate(objectives):
    for batch in range(12):
 

        image_prefix = obj * 24 + batch*2
        excel_vector_index = int(image_prefix / 2)

        # Load optimised data
        optimised_patterns_data = r'output\excel_docs'
        op_metrics, omeas_error, op_param, onans, oindicators = ipt.read_spec_excel(optimised_patterns_data, doc_num = 88,print_doc=False )
        # print('Opt error shape:',omeas_error.shape)

        # Load batch data
        excel_folder_address = rf'output\excel_docs\excel_{batch}'
        p_metrics, meas_error, p_param, nans, indicators = ipt.read_spec_excel(excel_folder_address, doc_num=None,print_doc=False)
        # print(f'Batch {batch} error shape: {meas_error.shape}')

        mask = (meas_error[:,0] != 0) & (~np.isnan(meas_error[:,0])) & (np.abs(np.nanmean(meas_error[:,0]) - meas_error[:,0]) <= 3*np.std(meas_error[:,0]))

        if analyse_error:
            error_column = 0
            opt_error = omeas_error[excel_vector_index,error_column]
            # Get percentile
            percentile_in_sample = 100 - stats.percentileofscore(meas_error[mask,error_column], opt_error, kind='strict')
            # print(opt_error)
            values_matrix[inde+1,batch+1] = float(np.round(opt_error, 6))
        else:
            opt_metric = op_metrics[excel_vector_index,inde]
            # Get percentile
            percentile_in_sample = stats.percentileofscore(p_metrics[mask,inde], opt_metric, kind='strict')


        
        # print(f"Percentile: {percentile_in_sample}")
        percentile_matrix[inde+1,batch+1] = float(np.round(percentile_in_sample,1))


df = pd.DataFrame(percentile_matrix[1:, :], columns=percentile_matrix[0, :])
vall = pd.DataFrame(values_matrix,)
print(df.to_string(index=False))
print('\n',vall.to_string(index=False))

In [ ]:
import numpy as np
import image_process_tool_box as ipt
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn import tree
from scipy.interpolate import RBFInterpolator
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_friedman2
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import DotProduct, WhiteKernel,RBF, Matern, ExpSineSquared, RationalQuadratic
from sklearn.gaussian_process.kernels import ConstantKernel as C, RationalQuadratic as RQ,ExpSineSquared as Exp
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import image_process_tool_box as ipt
from sklearn.preprocessing import StandardScaler
from sklearn.exceptions import ConvergenceWarning
import warnings
warnings.filterwarnings(action='ignore', category=ConvergenceWarning)

# Load class data
excel_path = r'output\excel_docs\excel_6'
p_metrics, meas_error, p_param, nans, indicators = ipt.read_spec_excel(
    excel_path, doc_num=None)

# Create the mask to filter out bad data points
errors = meas_error[:, 0]
mask = (~np.isnan(errors)) & (errors != 0) & (np.abs(errors - np.nanmean(errors)) <= 3 * np.nanstd(errors))

# Find the true best points from the filtered raw data
filtered_errors = errors[mask]
filtered_mig_values = p_metrics[mask, 1]

# Find the point with the minimum RMSE (the best error)
# true_best_rmse_index = np.argmin(filtered_errors)
true_best_rmse_index = np.argsort(filtered_errors)[(len(filtered_errors)//4)]
true_best_rmse = filtered_errors[true_best_rmse_index]

# Find the point with the maximum MIG (the best quality)
# true_best_mig_index = np.argmax(filtered_mig_values)
true_best_mig_index = np.argsort(filtered_mig_values)[(len(filtered_mig_values)//4)]
true_best_mig = filtered_mig_values[true_best_mig_index]

# Prepare data for modeling
x_1 = p_param[mask, 0]
x_2 = p_param[mask, 1]
x_3 = p_param[mask, 2]
x_4 = p_param[mask, 3]
x_5 = p_param[mask, 4]

X = np.column_stack([x_1, x_3, x_4, x_5])
y_rmse = np.abs(filtered_errors.reshape(-1, 1))
y_mig = filtered_mig_values.reshape(-1, 1)
# Corrected: x_6 is the nan ratio from the nans array
x_6 = nans[mask, 0]

# Split data
X_train, X_test, y_rmse_train, y_rmse_test = train_test_split(X, y_rmse, test_size=0.2, random_state=42)
_, _, y_mig_train, y_mig_test = train_test_split(X, y_mig, test_size=0.2, random_state=42)
_, _, x_6_train, x_6_test = train_test_split(X, x_6, test_size=0.2, random_state=42)


# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
scaler_rmse = StandardScaler()
y_rmse_train_scaled = scaler_rmse.fit_transform(y_rmse_train)
scaler_mig = StandardScaler()
y_mig_train_scaled = scaler_mig.fit_transform(y_mig_train)

# Train model
model_type = 0

if model_type == 0:
    print("Support vector regression:")
    p_grid = [
        {'kernel': ['rbf'], 
         'C': np.logspace(-3, 3, 7), 
         'gamma': ['scale', 'auto']}
         ]
    
    rmse_grid = GridSearchCV(SVR(), param_grid=p_grid, cv=5, n_jobs=-1, scoring='r2')
    rmse_grid.fit(X_train_scaled, y_rmse_train_scaled.ravel())

    # Train MIG model
    mig_grid = GridSearchCV(SVR(), param_grid=p_grid, cv=5, n_jobs=-1, scoring='r2')
    mig_grid.fit(X_train_scaled, y_mig_train_scaled.ravel())

    # Define constraint inequality
    scaler_nan = StandardScaler()
    x_6_train_scaled = scaler_nan.fit_transform(x_6_train.reshape(-1, 1)).ravel()

    svr_nan = SVR()
    grid_nan = GridSearchCV(svr_nan, param_grid=p_grid, cv=5, n_jobs=-1)
    grid_nan.fit(X_train_scaled, x_6_train_scaled)
    svr_nan = grid_nan.best_estimator_

elif model_type == 1:
    print("Gaussian process regression")
    
    p_grid = { 
        'kernel': [
            # ConstantKernel(),                 # constant
            DotProduct(),                     # linear
            DotProduct(sigma_0=1.0)**3,      # polynomial
            RBF(),                           # squared exponential
            Matern(nu=1.5),                  # matern
            Matern(nu=0.5),                  # exponential
            ExpSineSquared(),                # gamma-exponential (periodic)
            RationalQuadratic(),             # rational quadratic
        ],   
        'alpha': np.logspace(-10, -1, 10)
    } 
    rmse_grid = GridSearchCV(GaussianProcessRegressor(), 
        param_grid=p_grid, 
        cv = 5, 
        n_jobs=-1,
        scoring='r2')
    rmse_grid.fit(X_train_scaled, y_rmse_train_scaled.ravel())
    # Train MIG model
    mig_grid = GridSearchCV(GaussianProcessRegressor(), param_grid=p_grid, cv=5, n_jobs=-1, scoring='r2')
    mig_grid.fit(X_train_scaled, y_mig_train_scaled.ravel())
    # Define constraint inequality
    scaler_nan = StandardScaler()
    x_6_train_scaled = scaler_nan.fit_transform(x_6_train.reshape(-1, 1)).ravel()

    gauss_nan = GaussianProcessRegressor()
    grid_nan = GridSearchCV(gauss_nan, param_grid=p_grid, cv=5, n_jobs=-1)
    grid_nan.fit(X_train_scaled, x_6_train_scaled)
    svr_nan = grid_nan.best_estimator_

    

# Define the nan threshold for the constraint
nan_threshold = 10.0 # Placeholder value, adjust as needed

def g1(x):
    """
    Constraint function based on predicted nan ratio.
    
    Args:
        x: The 5 input parameters.
        
    Returns:
        The constraint value (Threshold - predicted_NAN_ratio(X)).
    """
    nan_scaled = svr_nan.predict(scaler.transform(x.reshape(1, -1)))
    nan_pred = scaler_nan.inverse_transform(nan_scaled.reshape(-1, 1)).ravel()[0]
    nan_pred = np.clip(nan_pred, 0, 100)
    # returns g(x) = Threshold - predicted_NAN_ratio(X) 
    return 50 - nan_pred


print("\nModels trained and ready for separate predictions.")

# Function to make a separate RMSE prediction
def predict_rmse(x1,x3, x4, x5):
    """
    Predicts the RMSE value for a given set of input parameters.
    
    Args:
        x1, x2, x3, x4, x5: The 5 input parameters.
        
    Returns:
        The predicted RMSE value.
    """
    input_params = np.array([[x1, x3, x4, x5]])
    input_scaled = scaler.transform(input_params)
    rmse_pred_scaled = rmse_grid.predict(input_scaled)[0]
    rmse_pred = scaler_rmse.inverse_transform([[rmse_pred_scaled]])[0][0]
    return rmse_pred

# Function to make a separate MIG prediction
def predict_mig(x1, x3, x4, x5):
    """
    Predicts the MIG value for a given set of input parameters.
    
    Args:
        x1, x2, x3, x4, x5: The 5 input parameters.
        
    Returns:
        The predicted MIG value.
    """
    input_params = np.array([[x1, x3, x4, x5]])
    input_scaled = scaler.transform(input_params)
    mig_pred_scaled = mig_grid.predict(input_scaled)[0]
    mig_pred = scaler_mig.inverse_transform([[mig_pred_scaled]])[0][0]
    return mig_pred

# --- Prediction and Difference Calculation Section ---
# This section uses the parameters from the data that resulted in the min RMSE and max MIG.

# Get the parameters associated with the true best RMSE and MIG
rmse_best_params = np.delete(p_param[mask][true_best_rmse_index], 1)  # remove column 2 (x2)
mig_best_params  = np.delete(p_param[mask][true_best_mig_index], 1)   # remove column 2 (x2)

# Make the predictions using these specific parameters
# predicted_rmse = predict_rmse(2.069,5.055,1.460,0.1004)

predicted_rmse = predict_rmse(*rmse_best_params)
predicted_mig  = predict_mig(*mig_best_params)

# Calculate the difference between the predicted and true values
rmse_difference = predicted_rmse - true_best_rmse
mig_difference = predicted_mig - true_best_mig

print("\n--- Prediction Results from Best Parameters ---")
print(f"Predicted RMSE: {predicted_rmse:.6f}")
print(f"Actual (True Best) RMSE: {true_best_rmse:.6f}")
print(f"Difference (Predicted - Actual) for RMSE: {rmse_difference:.6f}")

print(f"\nPredicted MIG: {predicted_mig:.4f}")
print(f"Actual (True Best) MIG: {true_best_mig:.4f}")
print(f"Difference (Predicted - Actual) for MIG: {mig_difference:.4f}")

# Demonstrate the use of the new constraint function
print("\n--- Constraint Function (g1) Check ---")
g1_for_rmse_params = g1(rmse_best_params)
g1_for_mig_params = g1(mig_best_params)
print(f"Constraint value (g1) for RMSE Best Parameters: {g1_for_rmse_params:.4f}")
print(f"Constraint value (g1) for MIG Best Parameters: {g1_for_mig_params:.4f}")


In [ ]:
import image_process_tool_box as ipt
import cv2
import os

pp = r"C:\Users\General User\nokop\pattern2\data\speckle_pattern_img\Optimised\optbatch_6\84_Generated_spec_image.tif"
imae = cv2.imread(pp,cv2.IMREAD_GRAYSCALE)

mig = ipt.MIG(imae)
print(mig)



rmse_pred = predict_rmse(2.979,3.294,1.391,6.608)
print(rmse_pred)

# 4. Subpixel displacement study

In [ ]:

''' 
Code for performing subpixel rigid body translation on multiple textured images at various displacments.
Currently, the code is set up to perform translations in the x-direction only. The code is yet to be
modulorized and the translation parameters are hard-coded. 

The goal is to make a self-contianed script that generates the set of translated images, performs DIC and 
evaluates the errors displaying them in the form of histograms and scatter plots.

The aim is to study the relationship between pattern contrast and morphology with DIC systematic (or bias) errors.
'''

# Ensure output directories exist
os.makedirs(subpixel_translation_path, exist_ok=True)
os.makedirs(disp_store, exist_ok=True)

# Translation parameters
''' The translation parameters are hard-coded. There is no translation in the y-direction.'''
u_0 = np.arange(0.0, 1.05, 0.05)  # x translations
v_0 = 0  # y translation
image_type = 'tif'

def get_reference_files(path):

    """Get sorted list of even-numbered reference files."""

    return sorted(
        (f for f in os.listdir(path) 
         if f.endswith(f".{image_type}") and f.split('_')[0].isdigit() 
         and int(f.split('_')[0]) % 2 == 0),
        key=lambda x: int(x.split('_')[0])
    )

def translate_image(image, u, v):

    """Apply translation to image using interpolation. Interpolation order can be set to 1 
    for rigid subpixel translation"""

    height, width = image.shape[:2]
    
    x, y = np.meshgrid(np.arange(width), np.arange(height))
    
    # Apply translation to grid
    x_displaced = x + u
    y_displaced = y + v
    
    new_coordinates = np.array([y_displaced,x_displaced ])
    # Interpolate
    translated = map_coordinates(image, new_coordinates, order=3)
    return translated.reshape(image.shape)



def gogo():
    ''' 
    Self-contained script for performing subpixel rigid body translation on multiple textured images at various displacments.
    Incorporates the functions: get_reference_files, translate_image, save_displacement and main.
    '''
    # Get reference files
    ref_files = get_reference_files(reference_image_path)
    
    for u in u_0:
        for ref_file in ref_files:

            u = round(u, 2)
            
            ref_number = int(ref_file.split('_')[0])
            print(f'Processing reference image {ref_number} with translation u={u}')
            
            # Load reference image
            ref_image_path = os.path.join(reference_image_path, ref_file)
            reference_image = cv2.imread(ref_image_path, cv2.IMREAD_GRAYSCALE)
            
            if reference_image is None:
                print(f'Failed to load image: {ref_image_path}')
                continue
                
            # Translate image
            translated_image = translate_image(reference_image, u, v_0)
            
            # Save translated image
            save_image = f'{ref_number}_{u}_Translated_spec_image.tif'
            translated_image_path = os.path.join(subpixel_translation_path, save_image)
            
            success = cv2.imwrite(translated_image_path, translated_image)
            if success:
                print(f'Saved translated image: {save_image}')
            else:
                print(f'Failed to save image: {translated_image_path}')


if __name__ == "__main__":
    gogo()

# 5. Error analysis (Load .npy)


In [ ]:
from pyNastran.bdf.bdf import BDF
from pyNastran.op2.op2 import OP2
import file_paths as path
from numpy.linalg import norm
import numpy as np
import matplotlib.pyplot as plt
import image_process_tool_box as ipt


# Load FEA data (Genesis)
op2_path = path.perf_30_5mm_op2
bdf_path = path.perf_30_5mm_bdf

nodes_2d, deformed_nodes_2d = ipt.load_fe_nodes(bdf_path=bdf_path, op2_path=op2_path)

# 3D plot of deformed_nodes_2d
# FEM data
fem_scale = 1000                                    # Scale up to match image coordinate system 
fem_xcoord = nodes_2d[:,0]*fem_scale
fem_ycoord = nodes_2d[:,1]*fem_scale
fem_x_disp = (deformed_nodes_2d[:,0] - nodes_2d[:,0])*fem_scale
fem_y_disp = (deformed_nodes_2d[:,1] - nodes_2d[:,1])*fem_scale
fem_lol = np.sqrt(fem_x_disp**2 + fem_y_disp**2)

# Load FEA data (NASTRAN)
top2_path = r"C:\Users\General User\nokop\Apex\NASTRAN\Processed_files\flat20_4mm.op2"
tbdf_path = r"C:\Users\General User\nokop\Apex\NASTRAN\Processed_files\flat20_4mm.bdf"

nasnodes_2d, nasdeformed_nodes_2d = ipt.load_fe_nodes(bdf_path=bdf_path, op2_path=op2_path)

# 3D plot of deformed_nodes_2d
# FEM data
fem_scale = 1000                                    # Scale up to match image coordinate system 
nasfem_xcoord = nasnodes_2d[:,0]*fem_scale
nasfem_ycoord = nasnodes_2d[:,1]*fem_scale
nasfem_x_disp = (nasdeformed_nodes_2d[:,0] - nasnodes_2d[:,0])*fem_scale
nasfem_y_disp = (nasdeformed_nodes_2d[:,1] - nasnodes_2d[:,1])*fem_scale
nasfem_lol = np.sqrt(fem_x_disp**2 + fem_y_disp**2)

displacement_difference = nasfem_x_disp - fem_x_disp
collapsed_errors = np.mean(nasfem_x_disp, axis = 0)

RMSe = np.mean(displacement_difference**2)

# Plot both
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
scatter = ax.scatter(fem_xcoord, fem_ycoord, fem_x_disp, c=fem_x_disp, cmap='jet', marker='o')
colorbar = plt.colorbar(scatter, ax=ax, label='Displacement magnitude [mm] (Genesis)')
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Displacement magnitude [mm]')
ax.set_title('FEM data')
plt.show()

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
scatter = ax.scatter(nasfem_xcoord, nasfem_ycoord, nasfem_x_disp, c=fem_x_disp, cmap='jet', marker='o')
colorbar = plt.colorbar(scatter, ax=ax, label='Displacement magnitude [mm] (NASTRAN)')
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Displacement magnitude [mm]')
ax.set_title('FEM data')
plt.show()

plt.scatter(fem_xcoord,displacement_difference)
plt.title("Displacement differences between Nastran and Genesis along x")
plt.show()

print('displacement_difference mean =', np.mean(displacement_difference))

In [ ]:
def displacement_continuity(dx: np.ndarray, dy: np.ndarray):
    """
    Check for discontinuities in the displacement field by looking at gradient.
    """
    # Calculate gradient of displacement field
    dx_grad_y, dx_grad_x = np.gradient(dx)
    dy_grad_y, dy_grad_x = np.gradient(dy)
    
    # Calculate gradient magnitude
    grad_magnitude = np.sqrt(dx_grad_x**2 + dx_grad_y**2 + 
                           dy_grad_x**2 + dy_grad_y**2)
    
    # Plot gradient magnitude to identify discontinuities
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(grad_magnitude, cmap='jet')
    ax.set_title('Displacement Field Gradient Magnitude')
    plt.colorbar(im, ax=ax)

    figg = ipt.fig_to_image(fig)

    return figg,grad_magnitude


dx1,dy1,_,_ = ipt.smooth_field(np.zeros((500,2000)), nasnodes_2d, nasdeformed_nodes_2d,kern =3)
fig1,gradmag = displacement_continuity(dx1,dy1)
plt.show()

dx2,dy2,_,_ = ipt.smooth_field(np.zeros((500,2000)), nodes_2d, deformed_nodes_2d, kern = 3)
fig2,gradmag2 = displacement_continuity(dx2,dy2)
plt.show()

print("mean strain magnitude difference", np.mean(gradmag2-gradmag))

In [ ]:
def show_strain_components(matrix, source_points, destination_points, path=None, pixel_size=1.0):
    """
    Function to visualize individual strain components for debugging
    """
    dx, dy, _, _ = ipt.smooth_field(matrix, source_points, destination_points, 3)
    
    # Calculate gradients
    dux_dy, dux_dx = np.gradient(dx, pixel_size, pixel_size)
    duy_dy, duy_dx = np.gradient(dy, pixel_size, pixel_size)
    
    # Strain components
    exx = dux_dx
    eyy = duy_dy  
    exy = 0.5 * (dux_dy + duy_dx)
    
    # Create subplot figure
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # Plot each component
    im1 = axes[0, 0].imshow(exx, cmap='RdBu_r')
    axes[0, 0].set_title('εxx (Normal strain in x)')
    plt.colorbar(im1, ax=axes[0, 0])
    
    im2 = axes[0, 1].imshow(eyy, cmap='RdBu_r')
    axes[0, 1].set_title('εyy (Normal strain in y)')
    plt.colorbar(im2, ax=axes[0, 1])
    
    im3 = axes[1, 0].imshow(exy, cmap='RdBu_r')
    axes[1, 0].set_title('εxy (Shear strain)')
    plt.colorbar(im3, ax=axes[1, 0])
    
    # Von Mises strain
    von_mises = np.sqrt(exx**2 + eyy**2 - exx*eyy + 3*exy**2)
    im4 = axes[1, 1].imshow(von_mises, cmap='viridis')
    axes[1, 1].set_title('von Mises strain')
    plt.colorbar(im4, ax=axes[1, 1])
    
    plt.tight_layout()
    
    if path is not None:
        fig_path = path
        name = 'Strain_Components.png'
        save_path = os.path.join(fig_path, name)
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
    return fig



fig_genesis = show_strain_components(np.zeros((500,2000)), nasnodes_2d, nasdeformed_nodes_2d)
plt.show()

fig_genesis = show_strain_components(np.zeros((500,2000)), nodes_2d, deformed_nodes_2d)
plt.show()


# 6. Heterogenous deformation

In [ ]:
import image_process_tool_box as ipt
import file_paths as path
import matplotlib.pyplot as plt
import numpy as np
from pyNastran.bdf.bdf import BDF
from pyNastran.op2.op2 import OP2

# Load FE field
op2_path = path.hole_2_op
bdf_path = path.hole_2_bdf

# op2_path = path.flat30_4mm_op2
# bdf_path = path.flat30_4mm_bdf

def my_load_fe_nodes(bdf_path, op2_path):

    """
    Loads FE data and scales it to an image. The aspect ratios of the image and the 
    FE data must be identical. Returns scaled vectors

    Uses class-based method of reading the respective files.

    Unscaled
    """
    model = OP2()
    model.read_op2(op2_path)
    bdf = BDF()
    bdf.read_bdf(bdf_path)

    # Extract the displacement data from the op2 file. Assuming isubcase = 1 and static analysis
    itime = 0
    isubcase = 1
    disp = model.displacements[isubcase]
    # Extract displacements and nodes
    txyz = disp.data[itime, :, :3]
    # nodes = np.array([bdf.nodes[nid].get_position() for nid in bdf.nodes])
    # Get corresponding node IDs in correct OP2 order
    disp_node_ids = disp.node_gridtype[:, 0]

    # Use GRID entries from BDF to get node coordinates
    nodes = np.array([bdf.nodes[nid].xyz for nid in disp_node_ids])

    # Assuming model is defined in xy extract only the x and y components
    nodes_2d = nodes[:, [0,2]]
    displacements_2d = txyz[:, [0,2]]

    deformed_nodes_2d = nodes_2d + displacements_2d

    return nodes_2d, deformed_nodes_2d


# nodes_2d, deformed_nodes_2d = my_load_fe_nodes(bdf_path=bdf_path, op2_path=op2_path)

nodes_2d, deformed_nodes_2d = ipt.load_fe_nodes(bdf_path=bdf_path, op2_path=op2_path)

x_scale = 1000

original_points = nodes_2d * x_scale
new_points = deformed_nodes_2d * x_scale


# Show points
points_mask = original_points[:,0] >= 250
print("original_points length:", len(original_points))
print("Masked points length:", len(original_points[points_mask]))

masked_orign = original_points[points_mask]

plt.figure(figsize=(12, 6))
# Original mesh
plt.subplot(1, 2, 1)
plt.scatter(masked_orign[:, 0] , masked_orign[:, 1], c='b', marker='o', label='Original')
plt.title('Original Mesh')
plt.xlabel('X')
plt.ylabel('Y')
plt.grid(True)
plt.axis('equal')
# Deformed mesh
plt.subplot(1, 2, 2)
plt.scatter(new_points[:, 0], new_points[:, 1], c='r', marker='o', label='Deformed')
plt.title('Deformed Mesh')
plt.xlabel('X')
plt.ylabel('Y')
plt.grid(True)
plt.axis('equal')
plt.show(block=False)


dx, dy, _, _ = ipt.smooth_field(
    np.zeros((500, 2000)),
    original_points,
    new_points,
    3
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import image_process_tool_box as ipt
import file_paths as path

# Dimensions
L = 2000  # mm (length in x)
H = 500   # mm (height in y)
t = 1     # mm (thickness)

# Material properties
E = 1e3  # MPa or N/mm² (Young's modulus)
P = 2000  # N (applied axial force)

# Grid definition
x = np.linspace(0, L, L)
y = np.linspace(0, H, H)
grid_x, grid_y = np.meshgrid(x, y)

# Cross-sectional area
A = t * H  # mm²

# Axial strain (constant in x)
strain_x = P / (A * E)  # unitless

# Axial displacement u(x) = ε * x
u = strain_x * grid_x  # mm

# No vertical displacement in ideal uniaxial tension
v = np.zeros_like(u)

# Visualisation of analytical field
plt.figure(figsize=(10, 4))
plt.contourf(grid_x, grid_y, u, levels=100, cmap='jet')
plt.title("Analytical horizontal displacement field (u)")
plt.xlabel("x [mm]")
plt.ylabel("y [mm]")
plt.colorbar(label='Displacement [mm]')
plt.show()


plt.figure(figsize=(10, 4))
plt.contourf(grid_x, grid_y, dx, levels=100, cmap='jet')
plt.title("Interpolated displacement field (u) ")
plt.xlabel("x [mm]")
plt.ylabel("y [mm]")
plt.colorbar(label='Displacement [mm]')
plt.show()


# -----------------------
# Error Calculation
# -----------------------

# Ensure dx and u have the same shape
assert dx.shape == u.shape, "Shape mismatch between interpolated and analytical fields"

# Calculate error
error = dx - u
rmse = np.sqrt(np.mean(error**2))
max_error = np.max(np.abs(error))

print(f"RMSE between analytical and interpolated dx field: {rmse:.6f} mm")
print(f"Maximum absolute error: {max_error:.6f} mm")

# -----------------------
# Plot error field
# -----------------------

plt.figure(figsize=(10, 4))
plt.contourf(grid_x, grid_y, error, levels=100, cmap='jet')
plt.title("Error field: interpolated dx − analytical u")
plt.xlabel("x [mm]")
plt.ylabel("y [mm]")
plt.colorbar(label='Error [mm]')
plt.show()

print("\n\n----------------------------")
print("STRAIN")

# -----------------------
# Compute Analytical and Interpolated Strain (∂u/∂x = ε_xx)
# -----------------------

# Grid spacing in x
dx_spacing = x[1] - x[0]

# Analytical strain: constant field
strain_xx_analytical = np.full_like(u, strain_x)

# Interpolated strain: numerical derivative of dx
strain_xx_interpolated = np.gradient(dx, dx_spacing, axis=1)

# -----------------------
# Plot Analytical Strain Field
# -----------------------
plt.figure(figsize=(10, 4))
plt.contourf(grid_x, grid_y, strain_xx_analytical, levels=100, cmap='jet')
plt.title("Analytical strain field (∂u/∂x)")
plt.xlabel("x [mm]")
plt.ylabel("y [mm]")
plt.colorbar(label='Strain εₓₓ')
plt.tight_layout()
plt.show()

# -----------------------
# Plot Interpolated Strain Field
# -----------------------
plt.figure(figsize=(10, 4))
plt.contourf(grid_x, grid_y, strain_xx_interpolated, levels=100, cmap='jet')
plt.title("Interpolated strain field ")
plt.xlabel("x [mm]")
plt.ylabel("y [mm]")
plt.colorbar(label='Strain εₓₓ')
plt.tight_layout()
plt.show()

# -----------------------
# Plot Error Field
# -----------------------
strain_error = strain_xx_interpolated - strain_xx_analytical
plt.figure(figsize=(10, 4))
plt.contourf(grid_x, grid_y, strain_error, levels=100, cmap='jet')
plt.title("Strain error field: ∂(dx)/∂x − ∂(u)/∂x")
plt.xlabel("x [mm]")
plt.ylabel("y [mm]")
plt.colorbar(label='Strain Error εₓₓ')
plt.tight_layout()
plt.show()

# -----------------------
# Print RMSE and Max Error
# -----------------------
strain_rmse = np.sqrt(np.mean(strain_error**2))
strain_max_error = np.max(np.abs(strain_error))
print(f"RMSE of strain field (∂u/∂x): {strain_rmse:.6e}")
print(f"Max absolute error in strain field: {strain_max_error:.6e}")




In [ ]:
import image_process_tool_box as ipt
import os
import cv2
import file_paths as path
import matplotlib.pyplot as plt
import numpy as np

op2_path = path.perf_30_57mm_op2
bdf_path = path.perf_30_57mm_bdf

op2_hole = r"C:\Users\General User\nokop\Apex\hhole.op2"
bdf_hole = r"C:\Users\General User\nokop\Apex\HHole.bdf"

# nodes_2d, deformed_nodes_2d = ipt.load_fe_nodes(bdf_path=bdf_path, op2_path=op2_path)
nodes_2d, deformed_nodes_2d  = ipt.load_fe_nodes(bdf_path=bdf_hole, op2_path=op2_hole)

scale = 1000
nodes_2d = nodes_2d * scale
deformed_nodes_2d = deformed_nodes_2d * scale
# holnodes_2d = holnodes_2d * scale
# holdeformed_nodes_2d = holdeformed_nodes_2d * scale

image_path = r"data\speckle_pattern_img\reference_im"
sort_files = sorted((file for file in os.listdir(image_path) 
                     if 
                    file.endswith('tif') and 
                    int(file.split('_')[0]) % 2 == 0),
                    key=lambda x: int(x.split('_')[0]))

print(sort_files)

image = ipt.readImage(os.path.join(image_path,sort_files[0]))

plt.imshow(image, cmap='grey')
plt.grid(False)
plt.show()

def centre_hole(image,radius = 150):

    center_x = 1000  # Second index is x
    center_y = int(image.shape[0] / 2)  
    cv2.circle(image, (center_x, center_y), radius, (0, 0, 0), -1)
    return image

hole = centre_hole(image)
plt.imshow(hole, cmap='grey')
plt.grid(False)
plt.imsave(os.path.join(image_path,'this.tiff'),hole)
plt.show()


dx1, dy1, _, _ = ipt.smooth_field(
            np.zeros((image.shape[0], image.shape[1])),
            nodes_2d,
            deformed_nodes_2d,
            1
        )


dx2, dy2, _, _ = ipt.smooth_field(
            np.zeros((image.shape[0], image.shape[1])),
            nodes_2d,
            deformed_nodes_2d,
            2
        )


dx3, dy3, _, _ = ipt.smooth_field(
            np.zeros((image.shape[0], image.shape[1])),
            nodes_2d,
            deformed_nodes_2d,
            3
        )


dx4, dy4, _, _ = ipt.smooth_field(
            np.zeros((image.shape[0], image.shape[1])),
            nodes_2d,
            deformed_nodes_2d,
            4
        )



transformed_image1, _ = ipt.cv2_deform(
                hole,
                dx=dx1,
                dy=dy1
            )


transformed_image4, _ = ipt.cv2_deform(
                hole,
                dx=dx4,
                dy=dy4
            )


image_error = np.abs(transformed_image1 - transformed_image4)


plt.hist(image_error.flatten(), bins=100, color='red')
plt.xlabel("Absolute pixel difference between images")
plt.ylabel("Number of pixels")
plt.show()
# # Display the original and transformed images using matplotlib
# # (with the FE mesh imposed)
# fig, axs = plt.subplots(2, 1, figsize=(12, 6))
# axs[0].imshow(hole, cmap='grey')
# axs[0].set_title(f"Reference")
# axs[0].scatter(
#     *zip(*original_points),
#     color='red',
#     s=5
# )  # Mark original points

# axs[1].imshow(transformed_image,cmap='grey')
# axs[1].set_title(f"Deformed")
# axs[1].scatter(
#     *zip(*new_points),
#     color='red',
#     s=5
# )  # Mark new points

# for ax in axs:
#     ax.axis('on')

# text = plt.figtext(
#     0.5,
#     0.02,
#     f'First image size: {hole.shape}, '
#     f'Second image: {transformed_image.shape}',
#     ha='center'
# )
# plt.show()

In [ ]:
import image_process_tool_box as ipt
import os
import cv2
import file_paths as path
import matplotlib.pyplot as plt
import numpy as np


# Read Nastran op2 files
op2_path = path.msjul_30_op2
bdf_path = path.msjul_30_bdf

nodes_2d, deformed_nodes_2d = ipt.load_fe_nodes(bdf_path=bdf_path, op2_path=op2_path)

np.save("undeformed.npy",nodes_2d)
np.save("deformed.npy",deformed_nodes_2d)

# Read Genesis op2 files
op2_path2 = path.genesis_30
bdf_path2 = path.msjul_30_bdf

nodes_2d2, deformed_nodes_2d2 = ipt.load_fe_nodes(bdf_path=bdf_path2, op2_path=op2_path2)

# Plot difference
plt.style.use(['science', 'no-latex','ieee', 'grid'])

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 5,
    'axes.labelsize': 5,
    'xtick.labelsize': 5,
    'ytick.labelsize': 5,
    'legend.fontsize': 7,
    'lines.linewidth': 1.25,
    'lines.markersize': 3
})

nodal_difference = (np.abs(deformed_nodes_2d2 - deformed_nodes_2d)).flatten()

coordinates = nodes_2d.flatten()


# nodal_difference_magnitude = np.linalg.norm(deformed_nodes_2d2*1 - deformed_nodes_2d*1, axis=1)

# plt.hist(nodal_difference_magnitude, bins=100, color='red')
# plt.xlabel("Absolute difference [mm]")
# plt.ylabel("Number of nodes")
# # plt.title(f"Nodal differences (Max: {np.max(nodal_difference_magnitude):.2e})")
# plt.show()


#### General image deformation

In [ ]:
from scipy.ndimage import map_coordinates
import numpy as np
from noise import pnoise2
# from pyNastran import bdf
# from pyNastran import op2

# Image deformation that uses displaced points. 
# There are two main methods: discrete and analytical deformation

# The mesh points are included as separate numpy bin files. Extracting them from BDF and OP2 files 
# requires the installation of the PyNastran library which I found to have dependents that conflict with 
# some of the SUN-DIC dependents ( one that comes to mind is Numpy but I think 
# Matplotlib gave me issues too. I think the biggest problem was with IPython but 
# i cant remember 
# how I fixed it, I just remember I had to change the import line in one of the 
# IPython-related packages)

original_points = np.load("undeformed.npy")
new_points = np.load("deformed.npy")

# DEFORMATION 1: Discrete deformation remaps pixels using scipy:
def deform_image(image, original_points, new_points, rbf_kernel=3):

    """
    Function assumes that image has the same aspect ratio as the
    imported points.

    """
    # Scale coordinates to image scale
    x_scale = image.shape[0] / (np.max(original_points[:,0])) 
    undeformed = original_points * x_scale
    deformed = new_points * x_scale     
    
    h, w = image.shape[:2]

    # pixel grid
    grid_y, grid_x = np.mgrid[0:h, 0:w]

    # displacement field interpolation using RBF interpolar
    # The thinplate spline is the most accurate (according to literature)
    # for complex fields but its also very slow.
    # I used cubic because it works just as well for simple deformation

    if dx is None or dy is None:

        kernel_options = {
            1: "thin_plate_spline",
            2: "linear",
            3: "cubic",
            4: "quintic"
        }
        kerne = kernel_options.get(rbf_kernel)
        if kerne is None:
            raise ValueError("Kernel must be 1, 2, 3, or 4.")

        grid = np.column_stack((grid_x.ravel(), grid_y.ravel()))

        # Get the displacements (deviations)
        disp_x = deformed[:, 0] - undeformed[:, 0]
        disp_y = deformed[:, 1] - undeformed[:, 1]

        rbf_x = RBFInterpolator(undeformed, disp_x, kernel=kerne)
        rbf_y = RBFInterpolator(undeformed, disp_y, kernel=kerne)
         
        # Unflatten to match image frame
        dx = rbf_x(grid).reshape(h, w)
        dy = rbf_y(grid).reshape(h, w)

    # coordinate transform
    # You can use an analytical function here instead of the FEM data
    # The function should just output dx and dy in the same shape as the image
    # maybe add the function here or make it separate and make this one take dx 
    # and dy as inputs
    new_x = grid_x - dx
    new_y = grid_y - dy
    new_coordinates = np.array([new_y, new_x])

    # Pixel remapping. OpenCV works just as well. 
    remapped_image = np.zeros_like(image)
    order = 3 # Set to cubic. No point in setting it higher than this.

    # For greyscale image
    if image.ndim == 2:
        remapped_image = map_coordinates(
            image, new_coordinates, order=order,
            mode="constant", prefilter=True
        )
    # Or multichannel image
    else:
        for imchannel in range(image.shape[2]):
            remapped_image[..., imchannel] = map_coordinates(
                image[..., imchannel], new_coordinates, order=order,
                mode="constant", prefilter=True
            )

    return remapped_image


# DEFORMATION 2: Analytical deformation samples a Perlin noise image on a displaced grid:
def deform_perlin_noise(h, w, dx_field, dy_field,
                         scale, octaves, persistence, lacunarity,
                         seed=None):
    """
    Generates one Perlin-noise reference image and its deformed counterpart.
    The deformation is applied by sampling the noise field at inverse-mapped
    coordinates defined by the displacement fields dx_field and dy_field.
    """

    if seed is not None:
        np.random.seed(seed)

    # Create coordinate grid. x and y store the pixel centre positions for a
    # regular h-by-w image. These integer coordinates are used both for noise
    # sampling and for evaluating the displacement fields.
    y, x = np.meshgrid(np.arange(h), np.arange(w), indexing='ij')
    coords = np.stack([x.flatten(), y.flatten()], axis=-1)

    # Evaluate displacements at each pixel location using the RBFInterpolator. 
    # You will notice that the function takes the actual interpolators as inputs
    # so the field interpolation process in the previous function will need to 
    # be performed outside the function. 
    
    # But like I said previously the dx and dy can come from an actual analytical funciton 
    # and not necessarily from the FEM data
    dx = dx_field(coords).reshape(h, w)
    dy = dy_field(coords).reshape(h, w)

    # Compute the inverse coordinates using dx and y
    x_inv = x - dx
    y_inv = y - dy

    # Build the reference (undeformed) Perlin image. 
    template = np.zeros((h, w), dtype=np.float32)
    for row in range(h):
        for col in range(w):
            template[row, col] = pnoise2(
                col * scale,
                row * scale,
                octaves=octaves,
                persistence=persistence,
                lacunarity=lacunarity,
                repeatx=w,
                repeaty=h,
                base=seed or 0
            )

    # Construct the deformed image by building the same image as above but this time 
    # on a shifted grid (src_x and src_y)
    deformed = np.zeros((h, w), dtype=np.float32)
    for row in range(h):
        for col in range(w):
            src_x = x_inv[row, col] * scale
            src_y = y_inv[row, col] * scale
            deformed[row, col] = pnoise2(
                src_x,
                src_y,
                octaves=octaves,
                persistence=persistence,
                lacunarity=lacunarity,
                repeatx=w,
                repeaty=h,
                base=seed or 0
            )

    # Convert both images to 8-bit greyscale and return
    template_8bit = ((template - template.min()) /
                     (template.max() - template.min()) * 255).astype(np.uint8)

    deformed_8bit = ((deformed - deformed.min()) /
                     (deformed.max() - deformed.min()) * 255).astype(np.uint8)

    return template_8bit, deformed_8bit




# Texture functions for 8 bit images
def apply_texture(image, texture_type="none"):
    """
    Applies various texture functions to Perlin noise images.
    
    Parameters:
        template      : Input image array
        texture_type  : Type of texture to apply
        modulate      : Whether to modulate the original image (not used in current implementation)
        scale         : Scale parameter for certain texture types
        seed          : Random seed for reproducibility
        pre_noise     : Pre-calculated phase noise for the leopard texture (to ensure consistency)
        
    Returns:
        Textured image as uint8 array

    Texture options:    none
                        thresholded
                        sinusoidal
                        bimodal
                        logarithmic
                        cubic
                        leopard ( to be added)
                        perlin_bandlimit
                      
    """
    if texture_type == "none":
        return image

    # Normalise 8bit image to [0,1]
    template = image.astype(np.float32) / 255.0

    # threshold (not to be confused with traditional binarising)
    if texture_type == "thresholded":
        return (template > 0.5).astype(np.uint8) * 255

    # (SInusoidal texture function)
    # Sinewave with a trough between 0 and 0.5 and a peak 
    # between 0.5 and 1. Grey values that are near the trough are suppressed
    # and those near the peak are emphasised.
    # The function is already scaled and shifted up to 255/2 so there is \
    # no need to multiply by 255
    elif texture_type == "sinusoidal":
        return (127.5 * (1 + np.sin(2 * np.pi * template))).astype(np.uint8)


   
    # similar intution as sine texture. Grey values near zero are affected heavily (most likely suppressed 
    # as that is where starts from zero with a very steep gradient)
     # to light very quickly but the change due to the texture function 
     # becomes progressively less intense as you go closer towards the 255 direction.
     # This was in the original 2006 paper.
    elif texture_type == "logarithmic":
        transformed = np.log(9 * template + 1)
        return (transformed * 255).astype(np.uint8)

   
    # Also from the paper. It might help to put the functions in desmos and 
    # see what they look like in x e[0,1] since the image is normalised to 
    # that range
    elif texture_type == "cubic":
        # T(x) = 32*{image - 0.5}^3 + 0.5
        transformed = 32.0 * (template - 0.5)**3 + 0.5
        transformed = np.clip(transformed, 0, 1)
        return (transformed * 255).astype(np.uint8)


   
    elif texture_type == "perlin_bandlimit":
        # Band-pass Perlin: keep values in a narrow range to isolate some structures
        lower_thresh = 0.4  
        upper_thresh = 0.6
        blobs = np.logical_and(template > lower_thresh, template < upper_thresh).astype(np.float32)
        return (blobs * 255).astype(np.uint8)
    
    # Apply a piecewise linear grey-level remapping: values below 0.5 are compressed,
    # and values above 0.5 are expanded, producing a bimodal grey-level distribution (didnt really use it much tbh)
    elif texture_type == "bimodal":
        bimodal = np.where(template < 0.5, template * 0.5, 0.5 + (template - 0.5) * 1.5)
        return (bimodal * 255).astype(np.uint8)

    # Leopard spots here. Where mentioned in the 2006 paper


    return (template * 255).astype(np.uint8)

